In [1]:
import requests
import json
import pandas as pd

opensky_api = "https://opensky-network.org/api/states/all"

opensky_data = requests.get(opensky_api).json()

opensky_columns = [
    "icao24",
    "callsign",
    "origin_country",
    "time_position",
    "last_contact",
    "longitude",
    "latitude",
    "baro_altitude",
    "on_ground",
    "velocity",
    "true_track",
    "vertical_rate",
    "sensors",
    "geo_altitude",
    "squawk",
    "spi",
    "position_source",
]

opensky_df = pd.DataFrame(opensky_data["states"], columns=opensky_columns)

In [2]:
sigmet_api = "https://aviationweather.gov/api/data/airsigmet"

params = {
    "format": "json"
}

sigmet_data = requests.get(sigmet_api, params=params).json()
sigmet_df = pd.DataFrame(sigmet_data)

In [3]:
import numpy as np
import gzip
import hashlib
from datetime import datetime, timezone

HEADERS = {
    "User-Agent": "Wilvor local schema notebook - contact: your_email@example.com"
}

def epoch_to_utc(series):
    return pd.to_datetime(series, unit="s", utc=True, errors="coerce")

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def make_hash_id(values, prefix):
    raw = "|".join([str(v) for v in values])
    return f"{prefix}_{hashlib.md5(raw.encode()).hexdigest()[:16]}"

def col(df, name, default=np.nan):
    if name in df.columns:
        return df[name]
    return pd.Series([default] * len(df), index=df.index)

def clean_callsign(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None

## Table 1: AircraftCurrentState

In [4]:
aircraft_current_state_df = pd.DataFrame()

aircraft_current_state_df["aircraft_id"] = col(opensky_df, "icao24").astype(str)
aircraft_current_state_df["callsign"] = col(opensky_df, "callsign").apply(clean_callsign)
aircraft_current_state_df["origin_country"] = col(opensky_df, "origin_country")

aircraft_current_state_df["position_time_epoch"] = col(opensky_df, "time_position")
aircraft_current_state_df["position_time_utc"] = epoch_to_utc(aircraft_current_state_df["position_time_epoch"])

aircraft_current_state_df["last_contact_epoch"] = col(opensky_df, "last_contact")
aircraft_current_state_df["last_contact_utc"] = epoch_to_utc(aircraft_current_state_df["last_contact_epoch"])

aircraft_current_state_df["latitude"] = col(opensky_df, "latitude")
aircraft_current_state_df["longitude"] = col(opensky_df, "longitude")

aircraft_current_state_df["baro_altitude_m"] = col(opensky_df, "baro_altitude")
aircraft_current_state_df["geo_altitude_m"] = col(opensky_df, "geo_altitude")

aircraft_current_state_df["baro_altitude_ft"] = aircraft_current_state_df["baro_altitude_m"] * 3.28084
aircraft_current_state_df["geo_altitude_ft"] = aircraft_current_state_df["geo_altitude_m"] * 3.28084

aircraft_current_state_df["ground_speed_mps"] = col(opensky_df, "velocity")
aircraft_current_state_df["ground_speed_kt"] = aircraft_current_state_df["ground_speed_mps"] * 1.94384

aircraft_current_state_df["track_deg"] = col(opensky_df, "true_track")
aircraft_current_state_df["vertical_rate_mps"] = col(opensky_df, "vertical_rate")
aircraft_current_state_df["vertical_rate_fpm"] = aircraft_current_state_df["vertical_rate_mps"] * 196.8504

aircraft_current_state_df["on_ground"] = col(opensky_df, "on_ground")
aircraft_current_state_df["squawk"] = col(opensky_df, "squawk")
aircraft_current_state_df["spi"] = col(opensky_df, "spi")
aircraft_current_state_df["position_source"] = col(opensky_df, "position_source")

aircraft_current_state_df["has_position"] = (
    aircraft_current_state_df["latitude"].notna()
    & aircraft_current_state_df["longitude"].notna()
)

aircraft_current_state_df["source_system"] = "OpenSky"
aircraft_current_state_df["schema_version"] = "aircraft_current_state.v1"
aircraft_current_state_df["received_at_utc"] = now_utc()

aircraft_current_state_df["idempotency_key"] = (
    aircraft_current_state_df["aircraft_id"].astype(str)
    + "#"
    + aircraft_current_state_df["position_time_epoch"].astype("Int64").astype(str)
)

aircraft_current_state_df.head()

,aircraft_id,callsign,origin_country,position_time_epoch,position_time_utc,last_contact_epoch,last_contact_utc,latitude,longitude,baro_altitude_m,...,vertical_rate_fpm,on_ground,squawk,spi,position_source,has_position,source_system,schema_version,received_at_utc,idempotency_key
0,ab1644,UAL521,United States,1.782599e+09,2026-06-27 22:24:24+00:00,1782599064,2026-06-27 22:24:24+00:00,29.9935,-95.0324,2026.92,...,-192.913392,False,NaN,False,0,True,OpenSky,aircraft_current_state.v1,2026-06-27T22:24:40.794173+00:00,ab1644#1782599064
1,80162c,AXB814,India,1.782599e+09,2026-06-27 22:24:24+00:00,1782599064,2026-06-27 22:24:24+00:00,18.1395,69.8412,10668.00,...,0.000000,False,NaN,False,0,True,OpenSky,aircraft_current_state.v1,2026-06-27T22:24:40.794173+00:00,80162c#1782599064
2,a1abec,N2069M,United States,1.782599e+09,2026-06-27 22:24:23+00:00,1782599063,2026-06-27 22:24:23+00:00,34.2055,-119.1087,243.84,...,-704.724432,False,NaN,False,0,True,OpenSky,aircraft_current_state.v1,2026-06-27T22:24:40.794173+00:00,a1abec#1782599063
3,aa9321,UAL933,United States,1.782599e+09,2026-06-27 22:24:23+00:00,1782599063,2026-06-27 22:24:23+00:00,46.7119,-70.1901,11582.40,...,0.000000,False,NaN,False,0,True,OpenSky,aircraft_current_state.v1,2026-06-27T22:24:40.794173+00:00,aa9321#1782599063
4,39de4c,TVF43PN,France,1.782599e+09,2026-06-27 22:24:24+00:00,1782599064,2026-06-27 22:24:24+00:00,40.4118,-3.4569,11871.96,...,0.000000,False,7665,False,0,True,OpenSky,aircraft_current_state.v1,2026-06-27T22:24:40.794173+00:00,39de4c#1782599064


## Table 2: ActiveHazards

In [5]:
active_hazards_df = pd.DataFrame()

sigmet_work = sigmet_df.reset_index(drop=True).copy()

active_hazards_df["hazard_id"] = sigmet_work.apply(
    lambda r: make_hash_id(
        [
            r.get("seriesId", r.name),
            r.get("creationTime"),
            r.get("validTimeFrom"),
            r.get("validTimeTo"),
            r.get("hazard")
        ],
        "hazard"
    ),
    axis=1
)

active_hazards_df["source_icao_id"] = col(sigmet_work, "icaoId")
active_hazards_df["series_id"] = col(sigmet_work, "seriesId")
active_hazards_df["alpha_char"] = col(sigmet_work, "alphaChar")

active_hazards_df["receipt_time_utc"] = pd.to_datetime(
    col(sigmet_work, "receiptTime"),
    utc=True,
    errors="coerce"
)

active_hazards_df["created_at_utc"] = pd.to_datetime(
    col(sigmet_work, "creationTime"),
    utc=True,
    errors="coerce"
)

active_hazards_df["valid_from_epoch"] = col(sigmet_work, "validTimeFrom")
active_hazards_df["valid_from_utc"] = epoch_to_utc(active_hazards_df["valid_from_epoch"])

active_hazards_df["valid_to_epoch"] = col(sigmet_work, "validTimeTo")
active_hazards_df["valid_to_utc"] = epoch_to_utc(active_hazards_df["valid_to_epoch"])

active_hazards_df["product_type"] = col(sigmet_work, "airSigmetType")
active_hazards_df["hazard_type"] = col(sigmet_work, "hazard")
active_hazards_df["severity"] = col(sigmet_work, "severity")

active_hazards_df["lower_altitude_1_ft"] = col(sigmet_work, "altitudeLow1")
active_hazards_df["upper_altitude_1_ft"] = col(sigmet_work, "altitudeHi1")

active_hazards_df["lower_altitude_2_ft"] = col(sigmet_work, "altitudeLow2")
active_hazards_df["upper_altitude_2_ft"] = col(sigmet_work, "altitudeHi2")

active_hazards_df["movement_direction_deg"] = col(sigmet_work, "movementDir")
active_hazards_df["movement_speed_kt"] = col(sigmet_work, "movementSpd")

active_hazards_df["polygon_coords"] = col(sigmet_work, "coords")
active_hazards_df["polygon_point_count"] = active_hazards_df["polygon_coords"].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

active_hazards_df["raw_text"] = col(sigmet_work, "rawAirSigmet")
active_hazards_df["post_process_flag"] = col(sigmet_work, "postProcessFlag")

active_hazards_df["status"] = "ACTIVE"
active_hazards_df["source_system"] = "NOAA_AviationWeather_SIGMET"
active_hazards_df["schema_version"] = "active_hazard.v1"
active_hazards_df["received_at_utc"] = now_utc()

active_hazards_df.head()

,hazard_id,source_icao_id,series_id,alpha_char,receipt_time_utc,created_at_utc,valid_from_epoch,valid_from_utc,valid_to_epoch,valid_to_utc,...,movement_direction_deg,movement_speed_kt,polygon_coords,polygon_point_count,raw_text,post_process_flag,status,source_system,schema_version,received_at_utc
0,hazard_01c38d64d506b14c,KSLC,NOVEMBER 1,N,2026-06-27 19:42:45.800000+00:00,2026-06-27 19:42:00+00:00,1782589320,2026-06-27 19:42:00+00:00,1782603720,2026-06-27 23:42:00+00:00,...,NaN,NaN,"[{'lat': 41.645, 'lon': -106.753}, {'lat': 40....",6,WSUS05 KKCI 271942\nSLCN WS 271942\nSIGMET NOV...,0,ACTIVE,NOAA_AviationWeather_SIGMET,active_hazard.v1,2026-06-27T22:24:40.871053+00:00
1,hazard_31e97ed6e53f5ad0,KKCI,08C,C,2026-06-27 21:50:21.183000+00:00,2026-06-27 21:55:00+00:00,1782597300,2026-06-27 21:55:00+00:00,1782604500,2026-06-27 23:55:00+00:00,...,250.0,25.0,"[{'lat': 39.81, 'lon': -86.37}, {'lat': 36.673...",6,WSUS32 KKCI 272155\nSIGC \nCONVECTIVE SIGMET 0...,0,ACTIVE,NOAA_AviationWeather_SIGMET,active_hazard.v1,2026-06-27T22:24:40.871053+00:00
2,hazard_7cdea978d605b74a,KKCI,09C,C,2026-06-27 21:50:21.215000+00:00,2026-06-27 21:55:00+00:00,1782597300,2026-06-27 21:55:00+00:00,1782604500,2026-06-27 23:55:00+00:00,...,240.0,20.0,"[{'lat': 47.714, 'lon': -96.841}, {'lat': 47.6...",5,WSUS32 KKCI 272155\nSIGC \nCONVECTIVE SIGMET 0...,0,ACTIVE,NOAA_AviationWeather_SIGMET,active_hazard.v1,2026-06-27T22:24:40.871053+00:00
3,hazard_45d636410dd9efc4,KKCI,10C,C,2026-06-27 21:50:21.246000+00:00,2026-06-27 21:55:00+00:00,1782597300,2026-06-27 21:55:00+00:00,1782604500,2026-06-27 23:55:00+00:00,...,200.0,35.0,"[{'lat': 43.722, 'lon': -103.86}, {'lat': 43.3...",5,WSUS32 KKCI 272155\nSIGC \nCONVECTIVE SIGMET 1...,0,ACTIVE,NOAA_AviationWeather_SIGMET,active_hazard.v1,2026-06-27T22:24:40.871053+00:00
4,hazard_bb81bbdc7686cfdd,KKCI,25E,E,2026-06-27 21:50:21.366000+00:00,2026-06-27 21:55:00+00:00,1782597300,2026-06-27 21:55:00+00:00,1782604500,2026-06-27 23:55:00+00:00,...,260.0,20.0,"[{'lat': 37.313, 'lon': -72.58}, {'lat': 35.05...",5,WSUS31 KKCI 272155\nSIGE \nCONVECTIVE SIGMET 2...,0,ACTIVE,NOAA_AviationWeather_SIGMET,active_hazard.v1,2026-06-27T22:24:40.871053+00:00


## Table 3: HazardCoordinates

In [6]:
hazard_coordinate_rows = []

for _, hazard in active_hazards_df.iterrows():
    coords = hazard["polygon_coords"]

    if not isinstance(coords, list):
        continue

    for sequence_number, point in enumerate(coords):
        hazard_coordinate_rows.append({
            "hazard_id": hazard["hazard_id"],
            "sequence_number": sequence_number,
            "latitude": point.get("lat"),
            "longitude": point.get("lon"),
            "valid_from_utc": hazard["valid_from_utc"],
            "valid_to_utc": hazard["valid_to_utc"],
            "hazard_type": hazard["hazard_type"],
            "severity": hazard["severity"],
            "schema_version": "hazard_coordinates.v1"
        })

hazard_coordinates_df = pd.DataFrame(hazard_coordinate_rows)

hazard_coordinates_df.head()

,hazard_id,sequence_number,latitude,longitude,valid_from_utc,valid_to_utc,hazard_type,severity,schema_version
0,hazard_01c38d64d506b14c,0,41.645,-106.753,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,TURB,5,hazard_coordinates.v1
1,hazard_01c38d64d506b14c,1,40.745,-105.024,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,TURB,5,hazard_coordinates.v1
2,hazard_01c38d64d506b14c,2,36.955,-104.629,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,TURB,5,hazard_coordinates.v1
3,hazard_01c38d64d506b14c,3,36.744,-106.856,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,TURB,5,hazard_coordinates.v1
4,hazard_01c38d64d506b14c,4,38.398,-109.684,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,TURB,5,hazard_coordinates.v1


## Table 4: StationReference

In [7]:
HEADERS = {
    "User-Agent": "Wilvor local schema notebook - contact: your_email@example.com"
}

STATION_CACHE_URL = "https://aviationweather.gov/data/cache/stations.cache.json.gz"

def load_station_reference():
    response = requests.get(STATION_CACHE_URL, headers=HEADERS, timeout=60)
    response.raise_for_status()

    raw = json.loads(gzip.decompress(response.content).decode("utf-8"))

    if isinstance(raw, list):
        rows = raw
    elif isinstance(raw, dict) and "features" in raw:
        rows = []
        for feature in raw["features"]:
            props = feature.get("properties", {}).copy()
            geom = feature.get("geometry", {})

            if geom.get("type") == "Point":
                coords = geom.get("coordinates", [])
                if len(coords) >= 2:
                    props["lon"] = coords[0]
                    props["lat"] = coords[1]

            rows.append(props)
    else:
        raise ValueError("Unexpected station reference format")

    station_raw_df = pd.json_normalize(rows)

    def first_existing(df, possible_cols):
        for c in possible_cols:
            if c in df.columns:
                return df[c]
        return pd.Series([np.nan] * len(df), index=df.index)

    station_reference_df = pd.DataFrame()

    station_reference_df["station_id"] = first_existing(
        station_raw_df,
        ["icaoId", "station_id", "stationId", "id", "ident"]
    ).astype(str).str.upper()

    station_reference_df["station_name"] = first_existing(
        station_raw_df,
        ["name", "site", "stationName"]
    )

    station_reference_df["latitude"] = pd.to_numeric(
        first_existing(station_raw_df, ["lat", "latitude"]),
        errors="coerce"
    )

    station_reference_df["longitude"] = pd.to_numeric(
        first_existing(station_raw_df, ["lon", "longitude"]),
        errors="coerce"
    )

    station_reference_df["elevation_m"] = pd.to_numeric(
        first_existing(station_raw_df, ["elev", "elevation", "elevation_m"]),
        errors="coerce"
    )

    station_reference_df = station_reference_df[
        station_reference_df["station_id"].notna()
        & station_reference_df["latitude"].notna()
        & station_reference_df["longitude"].notna()
    ].drop_duplicates("station_id").copy()

    station_reference_df["schema_version"] = "station_reference.v1"

    return station_reference_df


station_reference_df = load_station_reference()
station_reference_df.head()

,station_id,station_name,latitude,longitude,elevation_m,schema_version
360,ABANG,Alexandria Bay,44.550,-75.930,76.0,station_reference.v1
365,AGGA,Auki Arpt,-8.703,160.682,2.0,station_reference.v1
366,AGGC,Choiseul Bay Arpt,-6.712,156.396,1.0,station_reference.v1
367,AGGH,Honiara/Henderson Arpt,-9.430,160.047,6.0,station_reference.v1
368,AGGK,Kirakira Arpt,-10.450,161.898,23.0,station_reference.v1


## Table 5: HazardStationCandidates

In [8]:
try:
    from shapely.geometry import Point, Polygon
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shapely"])
    from shapely.geometry import Point, Polygon


def polygon_from_coords(coords):
    if not isinstance(coords, list) or len(coords) < 3:
        return None

    points = []

    for p in coords:
        lat = p.get("lat")
        lon = p.get("lon")

        if lat is None or lon is None:
            continue

        points.append((lon, lat))

    if len(points) < 3:
        return None

    poly = Polygon(points)

    if not poly.is_valid:
        poly = poly.buffer(0)

    return poly


def find_stations_near_sigmets(active_hazards_df, station_reference_df, buffer_degrees=0.75):
    rows = []

    for _, hazard in active_hazards_df.iterrows():
        poly = polygon_from_coords(hazard["polygon_coords"])

        if poly is None:
            continue

        search_area = poly.buffer(buffer_degrees)

        for _, station in station_reference_df.iterrows():
            point = Point(station["longitude"], station["latitude"])

            if search_area.contains(point):
                rows.append({
                    "hazard_id": hazard["hazard_id"],
                    "hazard_type": hazard["hazard_type"],
                    "severity": hazard["severity"],
                    "valid_from_utc": hazard["valid_from_utc"],
                    "valid_to_utc": hazard["valid_to_utc"],
                    "station_id": station["station_id"],
                    "station_name": station["station_name"],
                    "station_latitude": station["latitude"],
                    "station_longitude": station["longitude"],
                    "reason": "STATION_INSIDE_OR_NEAR_SIGMET",
                    "schema_version": "hazard_station_candidate.v1"
                })

    return pd.DataFrame(rows).drop_duplicates(["hazard_id", "station_id"])


hazard_station_candidates_df = find_stations_near_sigmets(
    active_hazards_df,
    station_reference_df,
    buffer_degrees=0.75
)

hazard_station_candidates_df.head()

,hazard_id,hazard_type,severity,valid_from_utc,valid_to_utc,station_id,station_name,station_latitude,station_longitude,reason,schema_version
0,hazard_01c38d64d506b14c,TURB,5,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,K04V,Saguache Muni,38.09900,-106.17000,STATION_INSIDE_OR_NEAR_SIGMET,hazard_station_candidate.v1
1,hazard_01c38d64d506b14c,TURB,5,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,K0CO,Berthoud Pass,39.79400,-105.76400,STATION_INSIDE_OR_NEAR_SIGMET,hazard_station_candidate.v1
2,hazard_01c38d64d506b14c,TURB,5,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,K1V1,Rifle,39.53000,-107.80000,STATION_INSIDE_OR_NEAR_SIGMET,hazard_station_candidate.v1
3,hazard_01c38d64d506b14c,TURB,5,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,K1V6,Canon City/Fremont Cnty Arpt,38.42559,-105.10477,STATION_INSIDE_OR_NEAR_SIGMET,hazard_station_candidate.v1
4,hazard_01c38d64d506b14c,TURB,5,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,K20V,Kremmling/McElroy Fld,40.05631,-106.37438,STATION_INSIDE_OR_NEAR_SIGMET,hazard_station_candidate.v1


## Table 6: METAR

In [9]:
sigmet_station_ids = (
    hazard_station_candidates_df["station_id"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

sigmet_station_ids[:20], len(sigmet_station_ids)

(['CWAQ',
  'CWKO',
  'CWVN',
  'K00U',
  'K04V',
  'K06U',
  'K0A9',
  'K0CO',
  'K0V1',
  'K0V4',
  'K0VG',
  'K0W8',
  'K10U',
  'K14Y',
  'K15J',
  'K18A',
  'K19A',
  'K1A5',
  'K1A6',
  'K1BM'],
 909)

In [10]:
METAR_API_URL = "https://aviationweather.gov/api/data/metar"

def chunk_list(items, chunk_size=100):
    for i in range(0, len(items), chunk_size):
        yield items[i:i + chunk_size]


def fetch_metar_for_ids(station_ids, chunk_size=100):
    station_ids = sorted(list(set([x for x in station_ids if pd.notna(x)])))

    if not station_ids:
        return pd.DataFrame()

    all_rows = []

    for chunk in chunk_list(station_ids, chunk_size):
        params = {
            "ids": ",".join(chunk),
            "format": "json"
        }

        response = requests.get(
            METAR_API_URL,
            params=params,
            headers=HEADERS,
            timeout=60
        )

        if response.status_code == 204:
            continue

        response.raise_for_status()

        data = response.json()

        if isinstance(data, list):
            all_rows.extend(data)
        elif isinstance(data, dict):
            all_rows.append(data)

    return pd.DataFrame(all_rows)


metar_df = fetch_metar_for_ids(sigmet_station_ids)

metar_df.head()

,icaoId,receiptTime,obsTime,reportTime,temp,dewp,wdir,wspd,visib,altim,...,lon,elev,name,cover,clouds,fltCat,precip,wgst,slp,presTend
0,K28J,2026-06-27T22:18:04.872Z,1782598500,2026-06-27T22:15:00.000Z,24.0,23.0,VRB,5.0,0.25,1018.7,...,-81.6918,9,"Palatka Muni, FL, US",OVC,"[{'cover': 'BKN', 'base': 300}, {'cover': 'BKN...",LIFR,NaN,NaN,NaN,NaN
1,K1S3,2026-06-27T22:19:06.802Z,1782598500,2026-06-27T22:15:00.000Z,22.8,17.7,90,7.0,10+,1000.1,...,-106.6196,831,"Forsyth/Tillitt Fld, MT, US",SCT,"[{'cover': 'FEW', 'base': 6000}, {'cover': 'SC...",VFR,0.005,NaN,NaN,NaN
2,K1U7,2026-06-27T22:19:06.817Z,1782598500,2026-06-27T22:15:00.000Z,16.0,5.0,290,8.0,10+,1010.6,...,-111.3370,1808,"Paris/Bear Lake Cnty Arpt, ID, US",BKN,"[{'cover': 'SCT', 'base': 5500}, {'cover': 'BK...",VFR,NaN,13.0,NaN,NaN
3,K24J,2026-06-27T22:20:11.319Z,1782598500,2026-06-27T22:15:00.000Z,31.0,24.0,0,0.0,10+,1018.4,...,-83.0225,30,"Live Oak/Suwannee Cnty, FL, US",CLR,[],VFR,NaN,NaN,NaN,NaN
4,K7W4,2026-06-27T22:18:12.891Z,1782598500,2026-06-27T22:15:00.000Z,26.2,23.0,0,0.0,10+,1015.0,...,-77.7453,115,"Bumpass/Lake Anna, VA, US",SCT,"[{'cover': 'SCT', 'base': 11000}]",VFR,NaN,NaN,NaN,NaN


## Table 7: MetarLatest

In [11]:
def col(df, name, default=np.nan):
    if name in df.columns:
        return df[name]
    return pd.Series([default] * len(df), index=df.index)


def epoch_to_utc(series):
    return pd.to_datetime(series, unit="s", utc=True, errors="coerce")


metar_latest_df = pd.DataFrame()

if len(metar_df) > 0:
    metar_latest_df["station_id"] = col(metar_df, "icaoId")
    metar_latest_df["station_name"] = col(metar_df, "name")

    metar_latest_df["observed_time_epoch"] = col(metar_df, "obsTime")
    metar_latest_df["observed_time_utc"] = epoch_to_utc(metar_latest_df["observed_time_epoch"])

    metar_latest_df["receipt_time_utc"] = pd.to_datetime(
        col(metar_df, "receiptTime"),
        utc=True,
        errors="coerce"
    )

    metar_latest_df["temperature_c"] = col(metar_df, "temp")
    metar_latest_df["dewpoint_c"] = col(metar_df, "dewp")

    metar_latest_df["wind_direction_deg"] = col(metar_df, "wdir")
    metar_latest_df["wind_speed_kt"] = col(metar_df, "wspd")
    metar_latest_df["wind_gust_kt"] = col(metar_df, "wgst")

    metar_latest_df["visibility_sm"] = col(metar_df, "visib")
    metar_latest_df["altimeter_hpa"] = col(metar_df, "altim")
    metar_latest_df["sea_level_pressure_hpa"] = col(metar_df, "slp")

    metar_latest_df["weather_string"] = col(metar_df, "wxString")
    metar_latest_df["precipitation_in"] = col(metar_df, "precip")
    metar_latest_df["flight_category"] = col(metar_df, "fltCat")
    metar_latest_df["metar_type"] = col(metar_df, "metarType")

    metar_latest_df["latitude"] = col(metar_df, "lat")
    metar_latest_df["longitude"] = col(metar_df, "lon")
    metar_latest_df["elevation_m"] = col(metar_df, "elev")

    metar_latest_df["clouds"] = col(metar_df, "clouds")
    metar_latest_df["raw_text"] = col(metar_df, "rawOb")

    metar_latest_df["source_system"] = "NOAA_AviationWeather_METAR"
    metar_latest_df["schema_version"] = "metar_latest.v1"

metar_latest_df.head()

,station_id,station_name,observed_time_epoch,observed_time_utc,receipt_time_utc,temperature_c,dewpoint_c,wind_direction_deg,wind_speed_kt,wind_gust_kt,...,precipitation_in,flight_category,metar_type,latitude,longitude,elevation_m,clouds,raw_text,source_system,schema_version
0,K28J,"Palatka Muni, FL, US",1782598500,2026-06-27 22:15:00+00:00,2026-06-27 22:18:04.872000+00:00,24.0,23.0,VRB,5.0,NaN,...,NaN,LIFR,METAR,29.6602,-81.6918,9,"[{'cover': 'BKN', 'base': 300}, {'cover': 'BKN...",METAR K28J 272215Z AUTO VRB05KT 1/4SM -TSRA FG...,NOAA_AviationWeather_METAR,metar_latest.v1
1,K1S3,"Forsyth/Tillitt Fld, MT, US",1782598500,2026-06-27 22:15:00+00:00,2026-06-27 22:19:06.802000+00:00,22.8,17.7,90,7.0,NaN,...,0.005,VFR,METAR,46.2695,-106.6196,831,"[{'cover': 'FEW', 'base': 6000}, {'cover': 'SC...",METAR K1S3 272215Z AUTO 09007KT 10SM FEW060 SC...,NOAA_AviationWeather_METAR,metar_latest.v1
2,K1U7,"Paris/Bear Lake Cnty Arpt, ID, US",1782598500,2026-06-27 22:15:00+00:00,2026-06-27 22:19:06.817000+00:00,16.0,5.0,290,8.0,13.0,...,NaN,VFR,METAR,42.2520,-111.3370,1808,"[{'cover': 'SCT', 'base': 5500}, {'cover': 'BK...",METAR K1U7 272215Z AUTO 29008G13KT 10SM SCT055...,NOAA_AviationWeather_METAR,metar_latest.v1
3,K24J,"Live Oak/Suwannee Cnty, FL, US",1782598500,2026-06-27 22:15:00+00:00,2026-06-27 22:20:11.319000+00:00,31.0,24.0,0,0.0,NaN,...,NaN,VFR,METAR,30.2990,-83.0225,30,[],METAR K24J 272215Z AUTO 00000KT 10SM CLR 31/24...,NOAA_AviationWeather_METAR,metar_latest.v1
4,K7W4,"Bumpass/Lake Anna, VA, US",1782598500,2026-06-27 22:15:00+00:00,2026-06-27 22:18:12.891000+00:00,26.2,23.0,0,0.0,NaN,...,NaN,VFR,METAR,37.9667,-77.7453,115,"[{'cover': 'SCT', 'base': 11000}]",METAR K7W4 272215Z AUTO 00000KT 10SM SCT110 26...,NOAA_AviationWeather_METAR,metar_latest.v1


## Table 8: TAF 

In [12]:
TAF_API_URL = "https://aviationweather.gov/api/data/taf"

def fetch_taf_for_ids(station_ids, chunk_size=100):
    station_ids = sorted(list(set([x for x in station_ids if pd.notna(x)])))

    if not station_ids:
        return pd.DataFrame()

    all_rows = []

    for chunk in chunk_list(station_ids, chunk_size):
        params = {
            "ids": ",".join(chunk),
            "format": "json"
        }

        response = requests.get(
            TAF_API_URL,
            params=params,
            headers=HEADERS,
            timeout=60
        )

        if response.status_code == 204:
            continue

        response.raise_for_status()

        data = response.json()

        if isinstance(data, list):
            all_rows.extend(data)
        elif isinstance(data, dict):
            all_rows.append(data)

    return pd.DataFrame(all_rows)


taf_df = fetch_taf_for_ids(sigmet_station_ids)

taf_df.head()

,icaoId,dbPopTime,bulletinTime,issueTime,validTimeFrom,validTimeTo,rawTAF,mostRecent,remarks,lat,lon,elev,prior,name,fcsts
0,KAHN,2026-06-27T22:11:58.767Z,2026-06-27T22:11:00.000Z,2026-06-27T22:11:00.000Z,1782597600,1782669600,TAF KAHN 272211Z 2722/2818 27008KT P6SM FEW030...,1,AMD,33.94773,-83.32736,241,3,Athens/Epps Arpt,"[{'timeFrom': 1782597600, 'timeTo': 1782608400..."
1,KAPF,2026-06-27T22:08:50.244Z,2026-06-27T22:08:00.000Z,2026-06-27T22:08:00.000Z,1782594000,1782669600,TAF KAPF 272208Z 2721/2818 25010KT P6SM SCT030...,1,COR,26.15498,-81.77513,2,4,Naples Muni,"[{'timeFrom': 1782594000, 'timeTo': 1782601200..."
2,KBKW,2026-06-27T22:02:29.890Z,2026-06-27T22:02:00.000Z,2026-06-27T22:02:00.000Z,1782597600,1782669600,TAF KBKW 272202Z 2722/2818 24008KT P6SM OVC040...,1,AMD,37.78359,-81.12282,762,3,Beckley/Raleigh Cnty,"[{'timeFrom': 1782597600, 'timeTo': 1782612000..."
3,KBYI,2026-06-27T21:58:40.423Z,2026-06-27T21:58:00.000Z,2026-06-27T21:58:00.000Z,1782597600,1782669600,TAF KBYI 272158Z 2722/2818 21010KT P6SM VCSH S...,1,AMD,42.54525,-113.76861,1263,4,Burley Muni,"[{'timeFrom': 1782597600, 'timeTo': 1782612000..."
4,KAND,2026-06-27T21:58:18.818Z,2026-06-27T21:58:00.000Z,2026-06-27T21:58:00.000Z,1782597600,1782669600,TAF KAND 272158Z 2722/2818 20011KT P6SM VCSH F...,1,AMD,34.49801,-82.70923,233,3,Anderson Rgnl,"[{'timeFrom': 1782597600, 'timeTo': 1782612000..."


## Table 9: TafLatest

In [13]:
taf_latest_df = pd.DataFrame()

if len(taf_df) > 0:
    taf_latest_df["station_id"] = col(taf_df, "icaoId")
    taf_latest_df["station_name"] = col(taf_df, "name")

    taf_latest_df["issued_at_utc"] = pd.to_datetime(
        col(taf_df, "issueTime"),
        utc=True,
        errors="coerce"
    )

    taf_latest_df["bulletin_time_utc"] = pd.to_datetime(
        col(taf_df, "bulletinTime"),
        utc=True,
        errors="coerce"
    )

    taf_latest_df["valid_from_epoch"] = col(taf_df, "validTimeFrom")
    taf_latest_df["valid_from_utc"] = epoch_to_utc(taf_latest_df["valid_from_epoch"])

    taf_latest_df["valid_to_epoch"] = col(taf_df, "validTimeTo")
    taf_latest_df["valid_to_utc"] = epoch_to_utc(taf_latest_df["valid_to_epoch"])

    taf_latest_df["most_recent"] = col(taf_df, "mostRecent")
    taf_latest_df["remarks"] = col(taf_df, "remarks")

    taf_latest_df["latitude"] = col(taf_df, "lat")
    taf_latest_df["longitude"] = col(taf_df, "lon")
    taf_latest_df["elevation_m"] = col(taf_df, "elev")

    taf_latest_df["raw_text"] = col(taf_df, "rawTAF")
    taf_latest_df["fcsts"] = col(taf_df, "fcsts")

    taf_latest_df["source_system"] = "NOAA_AviationWeather_TAF"
    taf_latest_df["schema_version"] = "taf_latest.v1"

taf_latest_df.head()

,station_id,station_name,issued_at_utc,bulletin_time_utc,valid_from_epoch,valid_from_utc,valid_to_epoch,valid_to_utc,most_recent,remarks,latitude,longitude,elevation_m,raw_text,fcsts,source_system,schema_version
0,KAHN,Athens/Epps Arpt,2026-06-27 22:11:00+00:00,2026-06-27 22:11:00+00:00,1782597600,2026-06-27 22:00:00+00:00,1782669600,2026-06-28 18:00:00+00:00,1,AMD,33.94773,-83.32736,241,TAF KAHN 272211Z 2722/2818 27008KT P6SM FEW030...,"[{'timeFrom': 1782597600, 'timeTo': 1782608400...",NOAA_AviationWeather_TAF,taf_latest.v1
1,KAPF,Naples Muni,2026-06-27 22:08:00+00:00,2026-06-27 22:08:00+00:00,1782594000,2026-06-27 21:00:00+00:00,1782669600,2026-06-28 18:00:00+00:00,1,COR,26.15498,-81.77513,2,TAF KAPF 272208Z 2721/2818 25010KT P6SM SCT030...,"[{'timeFrom': 1782594000, 'timeTo': 1782601200...",NOAA_AviationWeather_TAF,taf_latest.v1
2,KBKW,Beckley/Raleigh Cnty,2026-06-27 22:02:00+00:00,2026-06-27 22:02:00+00:00,1782597600,2026-06-27 22:00:00+00:00,1782669600,2026-06-28 18:00:00+00:00,1,AMD,37.78359,-81.12282,762,TAF KBKW 272202Z 2722/2818 24008KT P6SM OVC040...,"[{'timeFrom': 1782597600, 'timeTo': 1782612000...",NOAA_AviationWeather_TAF,taf_latest.v1
3,KBYI,Burley Muni,2026-06-27 21:58:00+00:00,2026-06-27 21:58:00+00:00,1782597600,2026-06-27 22:00:00+00:00,1782669600,2026-06-28 18:00:00+00:00,1,AMD,42.54525,-113.76861,1263,TAF KBYI 272158Z 2722/2818 21010KT P6SM VCSH S...,"[{'timeFrom': 1782597600, 'timeTo': 1782612000...",NOAA_AviationWeather_TAF,taf_latest.v1
4,KAND,Anderson Rgnl,2026-06-27 21:58:00+00:00,2026-06-27 21:58:00+00:00,1782597600,2026-06-27 22:00:00+00:00,1782669600,2026-06-28 18:00:00+00:00,1,AMD,34.49801,-82.70923,233,TAF KAND 272158Z 2722/2818 20011KT P6SM VCSH F...,"[{'timeFrom': 1782597600, 'timeTo': 1782612000...",NOAA_AviationWeather_TAF,taf_latest.v1


## Table 10: TafForecastPeriods

In [14]:
taf_period_rows = []

if len(taf_df) > 0 and "fcsts" in taf_df.columns:
    for _, taf in taf_df.iterrows():
        fcsts = taf.get("fcsts")

        if not isinstance(fcsts, list):
            continue

        for sequence_number, forecast in enumerate(fcsts):
            taf_period_rows.append({
                "station_id": taf.get("icaoId"),
                "issued_at_utc": pd.to_datetime(
                    taf.get("issueTime"),
                    utc=True,
                    errors="coerce"
                ),

                "period_from_epoch": forecast.get("timeFrom"),
                "period_from_utc": pd.to_datetime(
                    forecast.get("timeFrom"),
                    unit="s",
                    utc=True,
                    errors="coerce"
                ),

                "period_to_epoch": forecast.get("timeTo"),
                "period_to_utc": pd.to_datetime(
                    forecast.get("timeTo"),
                    unit="s",
                    utc=True,
                    errors="coerce"
                ),

                "change_type": forecast.get("fcstChange"),
                "probability": forecast.get("probability"),

                "wind_direction_deg": forecast.get("wdir"),
                "wind_speed_kt": forecast.get("wspd"),
                "wind_gust_kt": forecast.get("wgst"),

                "visibility_sm": forecast.get("visib"),
                "weather_string": forecast.get("wxString"),
                "clouds": forecast.get("clouds"),

                "sequence_number": sequence_number,
                "schema_version": "taf_forecast_period.v1"
            })

taf_forecast_periods_df = pd.DataFrame(taf_period_rows)

taf_forecast_periods_df.head()

,station_id,issued_at_utc,period_from_epoch,period_from_utc,period_to_epoch,period_to_utc,change_type,probability,wind_direction_deg,wind_speed_kt,wind_gust_kt,visibility_sm,weather_string,clouds,sequence_number,schema_version
0,KAHN,2026-06-27 22:11:00+00:00,1782597600,2026-06-27 22:00:00+00:00,1782608400,2026-06-28 01:00:00+00:00,NaN,NaN,270,8.0,NaN,6+,NaN,"[{'cover': 'FEW', 'base': 3000, 'type': None},...",0,taf_forecast_period.v1
1,KAHN,2026-06-27 22:11:00+00:00,1782597600,2026-06-27 22:00:00+00:00,1782601200,2026-06-27 23:00:00+00:00,TEMPO,NaN,270,15.0,25.0,6,-TSRA,"[{'cover': 'SCT', 'base': 4000, 'type': 'CB'},...",1,taf_forecast_period.v1
2,KAHN,2026-06-27 22:11:00+00:00,1782608400,2026-06-28 01:00:00+00:00,1782644400,2026-06-28 11:00:00+00:00,FM,NaN,250,4.0,NaN,6+,NaN,"[{'cover': 'FEW', 'base': 5000, 'type': None},...",2,taf_forecast_period.v1
3,KAHN,2026-06-27 22:11:00+00:00,1782644400,2026-06-28 11:00:00+00:00,1782669600,2026-06-28 18:00:00+00:00,FM,NaN,250,4.0,NaN,6+,NaN,"[{'cover': 'FEW', 'base': 800, 'type': None}, ...",3,taf_forecast_period.v1
4,KAPF,2026-06-27 22:08:00+00:00,1782594000,2026-06-27 21:00:00+00:00,1782601200,2026-06-27 23:00:00+00:00,NaN,NaN,250,10.0,NaN,6+,NaN,"[{'cover': 'SCT', 'base': 3000, 'type': None},...",0,taf_forecast_period.v1


## Table 11: HazardCells

In [15]:
def get_existing_table(*possible_names):
    for name in possible_names:
        if name in globals():
            return globals()[name].copy()
    raise NameError(f"None of these tables exist: {possible_names}")

AircraftCurrentState_df = get_existing_table(
    "AircraftCurrentState",
    "aircraft_current_state_df"
)

ActiveHazards_df = get_existing_table(
    "ActiveHazards",
    "active_hazards_df"
)

HazardCoordinates_df = get_existing_table(
    "HazardCoordinates",
    "hazard_coordinates_df"
)

StationReference_df = get_existing_table(
    "StationReference",
    "station_reference_df"
)

HazardStationCandidates_df = get_existing_table(
    "HazardStationCandidates",
    "hazard_station_candidates_df"
)

MetarLatest_df = get_existing_table(
    "MetarLatest",
    "metar_latest_df"
)

TafLatest_df = get_existing_table(
    "TafLatest",
    "taf_latest_df"
)

TafForecastPeriods_df = get_existing_table(
    "TafForecastPeriods",
    "taf_forecast_periods_df"
)

def now_utc():
    return datetime.now(timezone.utc)

def make_hash_id(values, prefix):
    raw = "|".join([str(v) for v in values])
    return f"{prefix}_{hashlib.md5(raw.encode()).hexdigest()[:16]}"

In [16]:
try:
    import h3
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "h3"])
    import h3

try:
    from shapely.geometry import Polygon, Point, LineString, mapping, shape
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shapely"])
    from shapely.geometry import Polygon, Point, LineString, mapping, shape


def h3_point(lat, lon, resolution=5):
    """
    Supports both h3-py v3 and v4 APIs.
    """
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lon, resolution)
    return h3.geo_to_h3(lat, lon, resolution)


def h3_polygon_cells_from_coords(coords, resolution=5):
    """
    coords should be a list of dicts:
    [{"lat": ..., "lon": ...}, ...]
    """
    if not isinstance(coords, list) or len(coords) < 3:
        return []

    ring = []

    for p in coords:
        lat = p.get("lat")
        lon = p.get("lon")

        if pd.notna(lat) and pd.notna(lon):
            ring.append([float(lon), float(lat)])

    if len(ring) < 3:
        return []

    if ring[0] != ring[-1]:
        ring.append(ring[0])

    geojson_polygon = {
        "type": "Polygon",
        "coordinates": [ring]
    }

    try:
        if hasattr(h3, "geo_to_cells"):
            return list(h3.geo_to_cells(geojson_polygon, resolution))
        return list(h3.polyfill(geojson_polygon, resolution, geo_json_conformant=True))
    except Exception:
        return []


H3_RESOLUTION = 5

hazard_cell_rows = []

for _, hazard in ActiveHazards_df.iterrows():
    coords = hazard.get("polygon_coords")

    cells = h3_polygon_cells_from_coords(coords, resolution=H3_RESOLUTION)

    for cell in cells:
        hazard_cell_rows.append({
            "h3_cell": cell,
            "hazard_id": hazard.get("hazard_id"),
            "hazard_type": hazard.get("hazard_type"),
            "severity": hazard.get("severity"),
            "valid_from_utc": hazard.get("valid_from_utc"),
            "valid_to_utc": hazard.get("valid_to_utc"),
            "schema_version": "hazard_cells.v1",
            "created_at_utc": now_utc()
        })

HazardCells = pd.DataFrame(hazard_cell_rows).drop_duplicates(
    ["h3_cell", "hazard_id"]
).reset_index(drop=True)

HazardCells.head()

,h3_cell,hazard_id,hazard_type,severity,valid_from_utc,valid_to_utc,schema_version,created_at_utc
0,85268e33fffffff,hazard_01c38d64d506b14c,TURB,5,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,hazard_cells.v1,2026-06-27 22:25:22.201162+00:00
1,85268ab7fffffff,hazard_01c38d64d506b14c,TURB,5,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,hazard_cells.v1,2026-06-27 22:25:22.201173+00:00
2,8548db2ffffffff,hazard_01c38d64d506b14c,TURB,5,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,hazard_cells.v1,2026-06-27 22:25:22.201179+00:00
3,85268043fffffff,hazard_01c38d64d506b14c,TURB,5,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,hazard_cells.v1,2026-06-27 22:25:22.201184+00:00
4,85269d2ffffffff,hazard_01c38d64d506b14c,TURB,5,2026-06-27 19:42:00+00:00,2026-06-27 23:42:00+00:00,hazard_cells.v1,2026-06-27 22:25:22.201190+00:00


In [17]:
HazardCells.shape

(5295, 8)

## Table 12: AircraftProjectionPoints

In [18]:
import math
import hashlib
from datetime import datetime, timezone, timedelta

def destination_point(lat, lon, bearing_deg, distance_nm):
    """
    Projects a lat/lon point using spherical earth math.
    distance_nm = nautical miles
    """
    radius_nm = 3440.065

    lat1 = math.radians(float(lat))
    lon1 = math.radians(float(lon))
    bearing = math.radians(float(bearing_deg))
    angular_distance = float(distance_nm) / radius_nm

    lat2 = math.asin(
        math.sin(lat1) * math.cos(angular_distance)
        + math.cos(lat1) * math.sin(angular_distance) * math.cos(bearing)
    )

    lon2 = lon1 + math.atan2(
        math.sin(bearing) * math.sin(angular_distance) * math.cos(lat1),
        math.cos(angular_distance) - math.sin(lat1) * math.sin(lat2)
    )

    return math.degrees(lat2), math.degrees(lon2)


def confidence_for_horizon(minutes):
    if minutes <= 5:
        return "HIGH"
    elif minutes <= 15:
        return "MEDIUM"
    return "LOW"


projection_rows = []

generated_at_utc = now_utc()

# For local testing, you can reduce this number if your OpenSky dataframe is huge.
MAX_AIRCRAFT_LOCAL = 1000

aircraft_for_projection = AircraftCurrentState_df.copy()

aircraft_for_projection = aircraft_for_projection[
    aircraft_for_projection["latitude"].notna()
    & aircraft_for_projection["longitude"].notna()
    & aircraft_for_projection["ground_speed_kt"].notna()
    & aircraft_for_projection["track_deg"].notna()
].copy()

if MAX_AIRCRAFT_LOCAL is not None:
    aircraft_for_projection = aircraft_for_projection.head(MAX_AIRCRAFT_LOCAL)

for _, ac in aircraft_for_projection.iterrows():
    aircraft_id = ac.get("aircraft_id")
    lat = ac.get("latitude")
    lon = ac.get("longitude")
    speed_kt = ac.get("ground_speed_kt")
    track_deg = ac.get("track_deg")
    vertical_rate_fpm = ac.get("vertical_rate_fpm", 0)

    altitude_ft = ac.get("geo_altitude_ft")

    if pd.isna(altitude_ft):
        altitude_ft = ac.get("baro_altitude_ft")

    if pd.isna(vertical_rate_fpm):
        vertical_rate_fpm = 0

    projection_id = make_hash_id(
        [aircraft_id, generated_at_utc.isoformat()],
        "projection"
    )

    for horizon_min in range(0, 31, 2):
        distance_nm = float(speed_kt) * horizon_min / 60.0

        projected_lat, projected_lon = destination_point(
            lat,
            lon,
            track_deg,
            distance_nm
        )

        projected_altitude_ft = None

        if pd.notna(altitude_ft):
            projected_altitude_ft = float(altitude_ft) + float(vertical_rate_fpm) * horizon_min
            projected_altitude_ft = max(projected_altitude_ft, 0)

        projection_rows.append({
            "projection_id": projection_id,
            "aircraft_id": aircraft_id,
            "generated_at_utc": generated_at_utc,
            "horizon_min": horizon_min,
            "projected_time_utc": generated_at_utc + timedelta(minutes=horizon_min),
            "latitude": projected_lat,
            "longitude": projected_lon,
            "estimated_altitude_ft": projected_altitude_ft,
            "confidence": confidence_for_horizon(horizon_min),
            "h3_cell": h3_point(projected_lat, projected_lon, resolution=H3_RESOLUTION),
            "schema_version": "aircraft_projection_point.v1"
        })

AircraftProjectionPoints = pd.DataFrame(projection_rows)

AircraftProjectionPoints.head()

,projection_id,aircraft_id,generated_at_utc,horizon_min,projected_time_utc,latitude,longitude,estimated_altitude_ft,confidence,h3_cell,schema_version
0,projection_2cda1b741a7ef3ab,ab1644,2026-06-27 22:25:22.295118+00:00,0,2026-06-27 22:25:22.295118+00:00,29.993500,-95.032400,7075.000226,HIGH,85446c77fffffff,aircraft_projection_point.v1
1,projection_2cda1b741a7ef3ab,ab1644,2026-06-27 22:25:22.295118+00:00,2,2026-06-27 22:27:22.295118+00:00,29.993985,-95.168294,6689.173442,HIGH,85446c2bfffffff,aircraft_projection_point.v1
2,projection_2cda1b741a7ef3ab,ab1644,2026-06-27 22:25:22.295118+00:00,4,2026-06-27 22:29:22.295118+00:00,29.994330,-95.304188,6303.346658,HIGH,85446c23fffffff,aircraft_projection_point.v1
3,projection_2cda1b741a7ef3ab,ab1644,2026-06-27 22:25:22.295118+00:00,6,2026-06-27 22:31:22.295118+00:00,29.994536,-95.440084,5917.519874,MEDIUM,85446c37fffffff,aircraft_projection_point.v1
4,projection_2cda1b741a7ef3ab,ab1644,2026-06-27 22:25:22.295118+00:00,8,2026-06-27 22:33:22.295118+00:00,29.994602,-95.575980,5531.693090,MEDIUM,85446c37fffffff,aircraft_projection_point.v1


## Table 13:AircraftProjection

In [19]:
def nm_to_degrees(nm):
    """
    Simple notebook approximation.
    1 degree latitude is about 60 nautical miles.
    """
    return nm / 60.0


def corridor_width_nm_for_projection(points_df):
    max_horizon = points_df["horizon_min"].max()

    if max_horizon <= 5:
        return 5
    elif max_horizon <= 15:
        return 10
    return 20


projection_summary_rows = []

for projection_id, group in AircraftProjectionPoints.groupby("projection_id"):
    group = group.sort_values("horizon_min").copy()

    aircraft_id = group["aircraft_id"].iloc[0]
    generated_at = group["generated_at_utc"].iloc[0]

    path_points = group[
        [
            "horizon_min",
            "projected_time_utc",
            "latitude",
            "longitude",
            "estimated_altitude_ft",
            "confidence",
            "h3_cell"
        ]
    ].to_dict("records")

    line_coords = list(zip(group["longitude"], group["latitude"]))

    corridor_polygon_geojson = None
    corridor_h3_cells = []

    if len(line_coords) >= 2:
        line = LineString(line_coords)

        corridor_half_width_nm = corridor_width_nm_for_projection(group)
        buffer_degrees = nm_to_degrees(corridor_half_width_nm)

        corridor_polygon = line.buffer(buffer_degrees)
        corridor_polygon_geojson = mapping(corridor_polygon)

        try:
            if hasattr(h3, "geo_to_cells"):
                corridor_h3_cells = list(h3.geo_to_cells(corridor_polygon_geojson, H3_RESOLUTION))
            else:
                corridor_h3_cells = list(
                    h3.polyfill(
                        corridor_polygon_geojson,
                        H3_RESOLUTION,
                        geo_json_conformant=True
                    )
                )
        except Exception:
            corridor_h3_cells = sorted(group["h3_cell"].dropna().unique().tolist())

    else:
        corridor_half_width_nm = None
        corridor_h3_cells = sorted(group["h3_cell"].dropna().unique().tolist())

    projection_summary_rows.append({
        "projection_id": projection_id,
        "aircraft_id": aircraft_id,
        "generated_at_utc": generated_at,
        "valid_until_utc": generated_at + timedelta(minutes=5),
        "projection_horizon_min": 30,
        "projection_points": path_points,
        "corridor_half_width_nm": corridor_half_width_nm,
        "corridor_polygon_geojson": corridor_polygon_geojson,
        "corridor_h3_cells": corridor_h3_cells,
        "corridor_h3_cell_count": len(corridor_h3_cells),
        "schema_version": "aircraft_projection.v1"
    })

AircraftProjection = pd.DataFrame(projection_summary_rows)

AircraftProjection.head()

,projection_id,aircraft_id,generated_at_utc,valid_until_utc,projection_horizon_min,projection_points,corridor_half_width_nm,corridor_polygon_geojson,corridor_h3_cells,corridor_h3_cell_count,schema_version
0,projection_0023a9ddeb4fa72a,3c4a0c,2026-06-27 22:25:22.295118+00:00,2026-06-27 22:30:22.295118+00:00,30,"[{'horizon_min': 0, 'projected_time_utc': 2026...",20,"{'type': 'Polygon', 'coordinates': (((-111.549...","[85299e53fffffff, 8529919bfffffff, 85269257fff...",103,aircraft_projection.v1
1,projection_002ae0806ad0b5f1,a53e35,2026-06-27 22:25:22.295118+00:00,2026-06-27 22:30:22.295118+00:00,30,"[{'horizon_min': 0, 'projected_time_utc': 2026...",20,"{'type': 'Polygon', 'coordinates': (((-83.5819...","[85441913fffffff, 8544e667fffffff, 85440223fff...",128,aircraft_projection.v1
2,projection_007ac69e7460538c,a9b4a4,2026-06-27 22:25:22.295118+00:00,2026-06-27 22:30:22.295118+00:00,30,"[{'horizon_min': 0, 'projected_time_utc': 2026...",20,"{'type': 'Polygon', 'coordinates': (((-84.5254...","[85264c1bfffffff, 85266953fffffff, 852669bbfff...",127,aircraft_projection.v1
3,projection_00954c7599f8cd41,47873f,2026-06-27 22:25:22.295118+00:00,2026-06-27 22:30:22.295118+00:00,30,"[{'horizon_min': 0, 'projected_time_utc': 2026...",20,"{'type': 'Polygon', 'coordinates': (((11.07245...","[85098a07fffffff, 85098bcbfffffff, 85098807fff...",97,aircraft_projection.v1
4,projection_00bb30d74d5d44ec,a0e1e8,2026-06-27 22:25:22.295118+00:00,2026-06-27 22:30:22.295118+00:00,30,"[{'horizon_min': 0, 'projected_time_utc': 2026...",20,"{'type': 'Polygon', 'coordinates': (((-117.293...","[85485d67fffffff, 85485cb3fffffff, 85485937fff...",118,aircraft_projection.v1


## Table 14: AircraftHazardCandidates

In [20]:
ProjectionCells = AircraftProjection[
    [
        "projection_id",
        "aircraft_id",
        "generated_at_utc",
        "corridor_h3_cells"
    ]
].explode("corridor_h3_cells").rename(columns={
    "corridor_h3_cells": "h3_cell"
})

ProjectionCells = ProjectionCells[
    ProjectionCells["h3_cell"].notna()
].drop_duplicates(
    ["projection_id", "h3_cell"]
)

AircraftHazardCandidates = ProjectionCells.merge(
    HazardCells[
        [
            "h3_cell",
            "hazard_id",
            "hazard_type",
            "severity",
            "valid_from_utc",
            "valid_to_utc"
        ]
    ],
    on="h3_cell",
    how="inner"
)

AircraftHazardCandidates = AircraftHazardCandidates.drop_duplicates(
    ["projection_id", "aircraft_id", "hazard_id"]
).reset_index(drop=True)

AircraftHazardCandidates["candidate_reason"] = "PROJECTION_CORRIDOR_CELL_OVERLAPS_HAZARD_CELL"
AircraftHazardCandidates["schema_version"] = "aircraft_hazard_candidate.v1"
AircraftHazardCandidates["created_at_utc"] = now_utc()

AircraftHazardCandidates.head()

,projection_id,aircraft_id,generated_at_utc,h3_cell,hazard_id,hazard_type,severity,valid_from_utc,valid_to_utc,candidate_reason,schema_version,created_at_utc
0,projection_0023a9ddeb4fa72a,3c4a0c,2026-06-27 22:25:22.295118+00:00,852692b3fffffff,hazard_e03eae59b7b971ab,CONVECTIVE,5,2026-06-27 21:55:00+00:00,2026-06-27 23:55:00+00:00,PROJECTION_CORRIDOR_CELL_OVERLAPS_HAZARD_CELL,aircraft_hazard_candidate.v1,2026-06-27 22:25:24.180057+00:00
1,projection_007ac69e7460538c,a9b4a4,2026-06-27 22:25:22.295118+00:00,85264c1bfffffff,hazard_31e97ed6e53f5ad0,CONVECTIVE,5,2026-06-27 21:55:00+00:00,2026-06-27 23:55:00+00:00,PROJECTION_CORRIDOR_CELL_OVERLAPS_HAZARD_CELL,aircraft_hazard_candidate.v1,2026-06-27 22:25:24.180057+00:00
2,projection_011e01a9256b9ba9,0d113b,2026-06-27 22:25:22.295118+00:00,8544db1bfffffff,hazard_92f1e5179770c450,CONVECTIVE,5,2026-06-27 21:55:00+00:00,2026-06-27 23:55:00+00:00,PROJECTION_CORRIDOR_CELL_OVERLAPS_HAZARD_CELL,aircraft_hazard_candidate.v1,2026-06-27 22:25:24.180057+00:00
3,projection_03b154e97f8c3c6f,ab6eb9,2026-06-27 22:25:22.295118+00:00,85441e4bfffffff,hazard_8cc4b3735942ae5e,CONVECTIVE,5,2026-06-27 21:55:00+00:00,2026-06-27 23:55:00+00:00,PROJECTION_CORRIDOR_CELL_OVERLAPS_HAZARD_CELL,aircraft_hazard_candidate.v1,2026-06-27 22:25:24.180057+00:00
4,projection_08468beec41f0132,0d1130,2026-06-27 22:25:22.295118+00:00,852a9937fffffff,hazard_92f1e5179770c450,CONVECTIVE,5,2026-06-27 21:55:00+00:00,2026-06-27 23:55:00+00:00,PROJECTION_CORRIDOR_CELL_OVERLAPS_HAZARD_CELL,aircraft_hazard_candidate.v1,2026-06-27 22:25:24.180057+00:00


## Table 15: HazardEncounters

In [21]:
def polygon_from_hazard_coords(coords):
    if not isinstance(coords, list) or len(coords) < 3:
        return None

    points = []

    for p in coords:
        lat = p.get("lat")
        lon = p.get("lon")

        if pd.notna(lat) and pd.notna(lon):
            points.append((float(lon), float(lat)))

    if len(points) < 3:
        return None

    poly = Polygon(points)

    if not poly.is_valid:
        poly = poly.buffer(0)

    return poly


def altitude_overlap(aircraft_alt_ft, lower_ft, upper_ft):
    if pd.isna(aircraft_alt_ft):
        return "UNKNOWN"

    if pd.isna(lower_ft) and pd.isna(upper_ft):
        return "UNKNOWN"

    if pd.isna(lower_ft):
        lower_ft = -999999

    if pd.isna(upper_ft):
        upper_ft = 999999

    return bool(lower_ft <= aircraft_alt_ft <= upper_ft)


def get_projection_points(projection_row):
    points = projection_row.get("projection_points")

    if isinstance(points, list):
        return pd.DataFrame(points)

    return pd.DataFrame()


encounter_rows = []

projection_lookup = AircraftProjection.set_index("projection_id")
hazard_lookup = ActiveHazards_df.set_index("hazard_id")

for _, candidate in AircraftHazardCandidates.iterrows():
    projection_id = candidate["projection_id"]
    hazard_id = candidate["hazard_id"]

    if projection_id not in projection_lookup.index:
        continue

    if hazard_id not in hazard_lookup.index:
        continue

    projection = projection_lookup.loc[projection_id]
    hazard = hazard_lookup.loc[hazard_id]

    hazard_polygon = polygon_from_hazard_coords(hazard.get("polygon_coords"))

    if hazard_polygon is None:
        continue

    corridor_geojson = projection.get("corridor_polygon_geojson")

    if corridor_geojson is None:
        continue

    try:
        corridor_polygon = shape(corridor_geojson)
    except Exception:
        continue

    corridor_intersects = corridor_polygon.intersects(hazard_polygon)

    if not corridor_intersects:
        continue

    projection_points = get_projection_points(projection)

    centerline_intersects = False
    inside_now = False
    first_horizon_min = None
    altitude_at_intersection_ft = None
    closest_distance_deg = None

    if len(projection_points) > 0:
        distances = []

        for _, point_row in projection_points.iterrows():
            p = Point(point_row["longitude"], point_row["latitude"])
            distance = p.distance(hazard_polygon)

            distances.append({
                "horizon_min": point_row["horizon_min"],
                "distance": distance,
                "estimated_altitude_ft": point_row.get("estimated_altitude_ft"),
                "inside": hazard_polygon.contains(p)
            })

        distance_df = pd.DataFrame(distances)

        inside_df = distance_df[distance_df["inside"] == True]

        if len(inside_df) > 0:
            centerline_intersects = True
            first_row = inside_df.sort_values("horizon_min").iloc[0]
        else:
            first_row = distance_df.sort_values("distance").iloc[0]

        first_horizon_min = first_row["horizon_min"]
        altitude_at_intersection_ft = first_row["estimated_altitude_ft"]
        closest_distance_deg = first_row["distance"]
        inside_now = bool(
            len(distance_df[
                (distance_df["horizon_min"] == 0)
                & (distance_df["inside"] == True)
            ]) > 0
        )

    lower_altitude_ft = hazard.get("lower_altitude_1_ft")
    upper_altitude_ft = hazard.get("upper_altitude_1_ft")

    alt_overlap = altitude_overlap(
        altitude_at_intersection_ft,
        lower_altitude_ft,
        upper_altitude_ft
    )

    encounter_id = make_hash_id(
        [
            projection.get("aircraft_id"),
            hazard_id,
            projection.get("generated_at_utc")
        ],
        "encounter"
    )

    encounter_rows.append({
        "encounter_id": encounter_id,
        "aircraft_id": projection.get("aircraft_id"),
        "projection_id": projection_id,
        "hazard_id": hazard_id,
        "hazard_type": hazard.get("hazard_type"),
        "severity": hazard.get("severity"),
        "detected_at_utc": now_utc(),
        "corridor_intersects": corridor_intersects,
        "centerline_intersects": centerline_intersects,
        "inside_now": inside_now,
        "first_intersection_horizon_min": first_horizon_min,
        "first_intersection_time_utc": (
            projection.get("generated_at_utc") + timedelta(minutes=float(first_horizon_min))
            if pd.notna(first_horizon_min)
            else None
        ),
        "altitude_at_intersection_ft": altitude_at_intersection_ft,
        "hazard_lower_altitude_ft": lower_altitude_ft,
        "hazard_upper_altitude_ft": upper_altitude_ft,
        "altitude_overlap": alt_overlap,
        "closest_distance_degrees": closest_distance_deg,
        "valid_from_utc": hazard.get("valid_from_utc"),
        "valid_to_utc": hazard.get("valid_to_utc"),
        "schema_version": "hazard_encounter.v1"
    })

HazardEncounters = pd.DataFrame(encounter_rows)

HazardEncounters.head()

,encounter_id,aircraft_id,projection_id,hazard_id,hazard_type,severity,detected_at_utc,corridor_intersects,centerline_intersects,inside_now,first_intersection_horizon_min,first_intersection_time_utc,altitude_at_intersection_ft,hazard_lower_altitude_ft,hazard_upper_altitude_ft,altitude_overlap,closest_distance_degrees,valid_from_utc,valid_to_utc,schema_version
0,encounter_6bf720e150754db2,3c4a0c,projection_0023a9ddeb4fa72a,hazard_e03eae59b7b971ab,CONVECTIVE,5,2026-06-27 22:25:24.205630+00:00,True,True,True,0,2026-06-27 22:25:22.295118+00:00,41200.001318,NaN,40000.0,False,0.0,2026-06-27 21:55:00+00:00,2026-06-27 23:55:00+00:00,hazard_encounter.v1
1,encounter_feada817d597fd86,a9b4a4,projection_007ac69e7460538c,hazard_31e97ed6e53f5ad0,CONVECTIVE,5,2026-06-27 22:25:24.208149+00:00,True,True,False,6,2026-06-27 22:31:22.295118+00:00,23514.764532,NaN,45000.0,True,0.0,2026-06-27 21:55:00+00:00,2026-06-27 23:55:00+00:00,hazard_encounter.v1
2,encounter_9dcffc7546003adc,0d113b,projection_011e01a9256b9ba9,hazard_92f1e5179770c450,CONVECTIVE,5,2026-06-27 22:25:24.210950+00:00,True,True,True,0,2026-06-27 22:25:22.295118+00:00,41300.001322,NaN,45000.0,True,0.0,2026-06-27 21:55:00+00:00,2026-06-27 23:55:00+00:00,hazard_encounter.v1
3,encounter_b60d3b0950945b93,ab6eb9,projection_03b154e97f8c3c6f,hazard_8cc4b3735942ae5e,CONVECTIVE,5,2026-06-27 22:25:24.213758+00:00,True,True,False,16,2026-06-27 22:41:22.295118+00:00,39125.001252,NaN,45000.0,True,0.0,2026-06-27 21:55:00+00:00,2026-06-27 23:55:00+00:00,hazard_encounter.v1
4,encounter_6d382ba1ad627b76,0d1130,projection_08468beec41f0132,hazard_92f1e5179770c450,CONVECTIVE,5,2026-06-27 22:25:24.215957+00:00,True,True,False,16,2026-06-27 22:41:22.295118+00:00,37900.001213,NaN,45000.0,True,0.0,2026-06-27 21:55:00+00:00,2026-06-27 23:55:00+00:00,hazard_encounter.v1


## Table 16: RiskResults

In [22]:
def hazard_component(hazard_type, severity):
    text = f"{hazard_type} {severity}".upper()

    if "CONVECTIVE" in text:
        return 35
    if "VOLCANIC" in text or "ASH" in text:
        return 35
    if "TURB" in text:
        return 30
    if "ICE" in text or "ICING" in text:
        return 30
    if "IFR" in text:
        return 25
    if "MTN" in text or "MOUNTAIN" in text:
        return 25

    return 20


def geometry_component(row):
    if row.get("inside_now") is True:
        return 30
    if row.get("centerline_intersects") is True:
        return 25
    if row.get("corridor_intersects") is True:
        return 15
    return 0


def time_component(minutes):
    if pd.isna(minutes):
        return 0

    if minutes <= 5:
        return 20
    if minutes <= 15:
        return 15
    if minutes <= 30:
        return 8

    return 0


def altitude_component(overlap):
    if overlap is True:
        return 15
    if overlap == "UNKNOWN":
        return 8
    return 0


def risk_level(score):
    if score >= 75:
        return "HIGH"
    if score >= 45:
        return "MEDIUM"
    return "LOW"


def build_reasons(row, score):
    reasons = []

    if row.get("inside_now") is True:
        reasons.append("Aircraft is currently inside the hazard polygon.")

    if row.get("centerline_intersects") is True:
        reasons.append("Projected centerline enters the active hazard polygon.")
    elif row.get("corridor_intersects") is True:
        reasons.append("Projected uncertainty corridor intersects the hazard polygon.")

    if pd.notna(row.get("first_intersection_horizon_min")):
        reasons.append(
            f"Estimated hazard proximity/intersection within {row.get('first_intersection_horizon_min')} minutes."
        )

    if row.get("altitude_overlap") is True:
        reasons.append("Estimated aircraft altitude overlaps the hazard altitude band.")
    elif row.get("altitude_overlap") == "UNKNOWN":
        reasons.append("Hazard altitude overlap is unknown because altitude bounds or aircraft altitude are incomplete.")
    else:
        reasons.append("Estimated aircraft altitude does not overlap the known hazard altitude band.")

    reasons.append(f"Calculated risk score is {score}.")

    return reasons


risk_rows = []

for _, encounter in HazardEncounters.iterrows():
    h_score = hazard_component(
        encounter.get("hazard_type"),
        encounter.get("severity")
    )

    g_score = geometry_component(encounter)
    t_score = time_component(encounter.get("first_intersection_horizon_min"))
    a_score = altitude_component(encounter.get("altitude_overlap"))

    total_score = h_score + g_score + t_score + a_score
    total_score = min(total_score, 100)

    level = risk_level(total_score)

    risk_id = make_hash_id(
        [
            encounter.get("aircraft_id"),
            encounter.get("hazard_id"),
            encounter.get("detected_at_utc")
        ],
        "risk"
    )

    risk_rows.append({
        "risk_id": risk_id,
        "encounter_id": encounter.get("encounter_id"),
        "aircraft_id": encounter.get("aircraft_id"),
        "hazard_id": encounter.get("hazard_id"),
        "hazard_type": encounter.get("hazard_type"),
        "severity": encounter.get("severity"),
        "risk_score": total_score,
        "risk_level": level,
        "hazard_component_score": h_score,
        "geometry_component_score": g_score,
        "time_component_score": t_score,
        "altitude_component_score": a_score,
        "confidence": "MEDIUM",
        "reasons": build_reasons(encounter, total_score),
        "generated_at_utc": now_utc(),
        "schema_version": "risk_result.v1"
    })

RiskResults = pd.DataFrame(risk_rows)

RiskResults.head()

,risk_id,encounter_id,aircraft_id,hazard_id,hazard_type,severity,risk_score,risk_level,hazard_component_score,geometry_component_score,time_component_score,altitude_component_score,confidence,reasons,generated_at_utc,schema_version
0,risk_e2ee33fe41c547b3,encounter_6bf720e150754db2,3c4a0c,hazard_e03eae59b7b971ab,CONVECTIVE,5,85,HIGH,35,30,20,0,MEDIUM,[Aircraft is currently inside the hazard polyg...,2026-06-27 22:25:24.608895+00:00,risk_result.v1
1,risk_d488c3cebf9a56e4,encounter_feada817d597fd86,a9b4a4,hazard_31e97ed6e53f5ad0,CONVECTIVE,5,90,HIGH,35,25,15,15,MEDIUM,[Projected centerline enters the active hazard...,2026-06-27 22:25:24.608985+00:00,risk_result.v1
2,risk_34a3027594d6f7a4,encounter_9dcffc7546003adc,0d113b,hazard_92f1e5179770c450,CONVECTIVE,5,100,HIGH,35,30,20,15,MEDIUM,[Aircraft is currently inside the hazard polyg...,2026-06-27 22:25:24.609051+00:00,risk_result.v1
3,risk_bec7d8b4adb1f3a2,encounter_b60d3b0950945b93,ab6eb9,hazard_8cc4b3735942ae5e,CONVECTIVE,5,83,HIGH,35,25,8,15,MEDIUM,[Projected centerline enters the active hazard...,2026-06-27 22:25:24.609104+00:00,risk_result.v1
4,risk_1f5ff0569e46fd13,encounter_6d382ba1ad627b76,0d1130,hazard_92f1e5179770c450,CONVECTIVE,5,83,HIGH,35,25,8,15,MEDIUM,[Projected centerline enters the active hazard...,2026-06-27 22:25:24.609160+00:00,risk_result.v1


## Table 17: AirportStatus

In [23]:
AirportStatus = StationReference_df.copy()

AirportStatus = AirportStatus.rename(columns={
    "station_id": "airport_id",
    "station_name": "airport_name"
})

metar_for_status = MetarLatest_df.copy().rename(columns={
    "station_id": "airport_id"
})

taf_for_status = TafLatest_df.copy().rename(columns={
    "station_id": "airport_id"
})

metar_cols = [
    "airport_id",
    "observed_time_utc",
    "temperature_c",
    "dewpoint_c",
    "wind_direction_deg",
    "wind_speed_kt",
    "wind_gust_kt",
    "visibility_sm",
    "weather_string",
    "flight_category",
    "raw_text"
]

metar_cols = [c for c in metar_cols if c in metar_for_status.columns]

AirportStatus = AirportStatus.merge(
    metar_for_status[metar_cols],
    on="airport_id",
    how="left"
)

taf_cols = [
    "airport_id",
    "issued_at_utc",
    "valid_from_utc",
    "valid_to_utc",
    "raw_text"
]

taf_cols = [c for c in taf_cols if c in taf_for_status.columns]

AirportStatus = AirportStatus.merge(
    taf_for_status[taf_cols],
    on="airport_id",
    how="left",
    suffixes=("_metar", "_taf")
)


def classify_airport_weather(row):
    flt_cat = str(row.get("flight_category", "")).upper()
    wx = str(row.get("weather_string", "")).upper()

    wind = row.get("wind_speed_kt")
    gust = row.get("wind_gust_kt")
    vis = row.get("visibility_sm")

    if "TS" in wx or "FZRA" in wx:
        return "HIGH"

    if flt_cat in ["LIFR", "IFR"]:
        return "HIGH"

    if pd.notna(gust) and float(gust) >= 35:
        return "HIGH"

    if pd.notna(wind) and float(wind) >= 30:
        return "MEDIUM"

    if pd.notna(vis) and str(vis) != "":
        try:
            if float(vis) < 3:
                return "HIGH"
            if float(vis) < 5:
                return "MEDIUM"
        except Exception:
            pass

    if flt_cat == "MVFR":
        return "MEDIUM"

    if flt_cat == "VFR":
        return "LOW"

    return "UNKNOWN"


AirportStatus["weather_risk_level"] = AirportStatus.apply(
    classify_airport_weather,
    axis=1
)

AirportStatus["has_metar"] = AirportStatus["observed_time_utc"].notna() if "observed_time_utc" in AirportStatus.columns else False
AirportStatus["has_taf"] = AirportStatus["issued_at_utc"].notna() if "issued_at_utc" in AirportStatus.columns else False

AirportStatus["schema_version"] = "airport_status.v1"
AirportStatus["updated_at_utc"] = now_utc()

AirportStatus.head()

,airport_id,airport_name,latitude,longitude,elevation_m,schema_version,observed_time_utc,temperature_c,dewpoint_c,wind_direction_deg,...,flight_category,raw_text_metar,issued_at_utc,valid_from_utc,valid_to_utc,raw_text_taf,weather_risk_level,has_metar,has_taf,updated_at_utc
0,ABANG,Alexandria Bay,44.550,-75.930,76.0,airport_status.v1,NaT,NaN,NaN,NaN,...,NaN,NaN,NaT,NaT,NaT,NaN,UNKNOWN,False,False,2026-06-27 22:25:24.728006+00:00
1,AGGA,Auki Arpt,-8.703,160.682,2.0,airport_status.v1,NaT,NaN,NaN,NaN,...,NaN,NaN,NaT,NaT,NaT,NaN,UNKNOWN,False,False,2026-06-27 22:25:24.728006+00:00
2,AGGC,Choiseul Bay Arpt,-6.712,156.396,1.0,airport_status.v1,NaT,NaN,NaN,NaN,...,NaN,NaN,NaT,NaT,NaT,NaN,UNKNOWN,False,False,2026-06-27 22:25:24.728006+00:00
3,AGGH,Honiara/Henderson Arpt,-9.430,160.047,6.0,airport_status.v1,NaT,NaN,NaN,NaN,...,NaN,NaN,NaT,NaT,NaT,NaN,UNKNOWN,False,False,2026-06-27 22:25:24.728006+00:00
4,AGGK,Kirakira Arpt,-10.450,161.898,23.0,airport_status.v1,NaT,NaN,NaN,NaN,...,NaN,NaN,NaT,NaT,NaT,NaN,UNKNOWN,False,False,2026-06-27 22:25:24.728006+00:00


## Table 18: AirportAssessment

In [24]:
def haversine_nm(lat1, lon1, lat2, lon2):
    radius_nm = 3440.065

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return radius_nm * c


def airport_weather_score(weather_risk_level):
    if weather_risk_level == "LOW":
        return 100
    if weather_risk_level == "MEDIUM":
        return 60
    if weather_risk_level == "HIGH":
        return 20
    return 40


assessment_rows = []

risky_aircraft = RiskResults[
    RiskResults["risk_level"].isin(["HIGH", "MEDIUM"])
].copy()

MAX_RISK_RESULTS_LOCAL = 100
risky_aircraft = risky_aircraft.head(MAX_RISK_RESULTS_LOCAL)

aircraft_lookup = AircraftCurrentState_df.set_index("aircraft_id")

for _, risk in risky_aircraft.iterrows():
    aircraft_id = risk["aircraft_id"]

    if aircraft_id not in aircraft_lookup.index:
        continue

    ac = aircraft_lookup.loc[aircraft_id]

    ac_lat = ac.get("latitude")
    ac_lon = ac.get("longitude")
    ac_speed = ac.get("ground_speed_kt")

    if pd.isna(ac_lat) or pd.isna(ac_lon):
        continue

    candidate_airports = AirportStatus.copy()

    candidate_airports = candidate_airports[
        candidate_airports["latitude"].notna()
        & candidate_airports["longitude"].notna()
    ].copy()

    candidate_airports["distance_nm"] = haversine_nm(
        float(ac_lat),
        float(ac_lon),
        candidate_airports["latitude"].astype(float),
        candidate_airports["longitude"].astype(float)
    )

    candidate_airports = candidate_airports[
        candidate_airports["distance_nm"] <= 250
    ].copy()

    candidate_airports = candidate_airports.sort_values("distance_nm").head(10)

    for _, airport in candidate_airports.iterrows():
        distance_nm = airport.get("distance_nm")

        eta_minutes = None

        if pd.notna(ac_speed) and float(ac_speed) > 0:
            eta_minutes = distance_nm / float(ac_speed) * 60

        weather_score = airport_weather_score(
            airport.get("weather_risk_level")
        )

        distance_score = max(0, 100 - distance_nm / 250 * 100)

        # MVP placeholder. Later this comes from route geometry and congestion logic.
        hazard_free_route_score = 60
        congestion_score = 50
        runway_score = 50

        total_score = (
            weather_score * 0.45
            + distance_score * 0.25
            + hazard_free_route_score * 0.15
            + congestion_score * 0.10
            + runway_score * 0.05
        )

        assessment_id = make_hash_id(
            [
                aircraft_id,
                risk.get("hazard_id"),
                airport.get("airport_id"),
                risk.get("generated_at_utc")
            ],
            "airport_assessment"
        )

        assessment_rows.append({
            "airport_assessment_id": assessment_id,
            "risk_id": risk.get("risk_id"),
            "aircraft_id": aircraft_id,
            "hazard_id": risk.get("hazard_id"),
            "airport_id": airport.get("airport_id"),
            "airport_name": airport.get("airport_name"),
            "airport_latitude": airport.get("latitude"),
            "airport_longitude": airport.get("longitude"),
            "distance_nm": distance_nm,
            "eta_minutes": eta_minutes,
            "weather_risk_level": airport.get("weather_risk_level"),
            "weather_score": weather_score,
            "distance_score": distance_score,
            "hazard_free_route_score": hazard_free_route_score,
            "congestion_score": congestion_score,
            "runway_score": runway_score,
            "total_airport_score": round(total_score, 2),
            "known_limitations": [
                "Runway suitability is placeholder only.",
                "Fuel endurance is unknown.",
                "Aircraft performance limits are unknown.",
                "Active runway is unknown.",
                "Airline policy is unknown."
            ],
            "schema_version": "airport_assessment.v1",
            "created_at_utc": now_utc()
        })

AirportAssessment = pd.DataFrame(assessment_rows)

AirportAssessment.head()

,airport_assessment_id,risk_id,aircraft_id,hazard_id,airport_id,airport_name,airport_latitude,airport_longitude,distance_nm,eta_minutes,weather_risk_level,weather_score,distance_score,hazard_free_route_score,congestion_score,runway_score,total_airport_score,known_limitations,schema_version,created_at_utc
0,airport_assessment_af078e59971b75db,risk_e2ee33fe41c547b3,3c4a0c,hazard_e03eae59b7b971ab,KPVU,Provo Muni,40.22403,-111.72524,5.878560,0.828132,LOW,100,97.648576,60,50,50,85.91,"[Runway suitability is placeholder only., Fuel...",airport_assessment.v1,2026-06-27 22:25:24.759463+00:00
1,airport_assessment_fe833c905f334f83,risk_e2ee33fe41c547b3,3c4a0c,hazard_e03eae59b7b971ab,KSPK,Spanish Fork Muni,40.14966,-111.67700,9.150108,1.289005,LOW,100,96.339957,60,50,50,85.58,"[Runway suitability is placeholder only., Fuel...",airport_assessment.v1,2026-06-27 22:25:24.759535+00:00
2,airport_assessment_c22273d22dab50de,risk_e2ee33fe41c547b3,3c4a0c,hazard_e03eae59b7b971ab,KHCR,Heber City Muni,40.47743,-111.43293,14.384694,2.026419,LOW,100,94.246122,60,50,50,85.06,"[Runway suitability is placeholder only., Fuel...",airport_assessment.v1,2026-06-27 22:25:24.759595+00:00
3,airport_assessment_20aa3a3133cc92df,risk_e2ee33fe41c547b3,3c4a0c,hazard_e03eae59b7b971ab,K36U,Heber/McDonald Fld,40.48300,-111.42700,14.814333,2.086943,UNKNOWN,40,94.074267,60,50,50,58.02,"[Runway suitability is placeholder only., Fuel...",airport_assessment.v1,2026-06-27 22:25:24.759875+00:00
4,airport_assessment_4006c42a3a196a3b,risk_e2ee33fe41c547b3,3c4a0c,hazard_e03eae59b7b971ab,KU42,Salt Lake City Muni,40.61960,-111.99019,24.857349,3.501736,UNKNOWN,40,90.057061,60,50,50,57.01,"[Runway suitability is placeholder only., Fuel...",airport_assessment.v1,2026-06-27 22:25:24.760439+00:00


## Table 19: Recommendations

In [25]:
recommendation_rows = []

for _, risk in RiskResults.iterrows():
    risk_id = risk.get("risk_id")
    aircraft_id = risk.get("aircraft_id")
    hazard_id = risk.get("hazard_id")
    risk_level_value = risk.get("risk_level")

    airport_options = AirportAssessment[
        AirportAssessment["risk_id"] == risk_id
    ].sort_values("total_airport_score", ascending=False)

    primary_action_type = "MONITOR"

    if risk_level_value == "HIGH":
        primary_action_type = "EVALUATE_AVOIDANCE_OR_DIVERSION"
    elif risk_level_value == "MEDIUM":
        primary_action_type = "MONITOR_AND_PREPARE_OPTIONS"

    preferred_airport_id = None
    preferred_airport_score = None
    candidate_airports = []

    if len(airport_options) > 0:
        best = airport_options.iloc[0]
        preferred_airport_id = best.get("airport_id")
        preferred_airport_score = best.get("total_airport_score")

        candidate_airports = airport_options[
            [
                "airport_id",
                "airport_name",
                "distance_nm",
                "eta_minutes",
                "weather_risk_level",
                "total_airport_score"
            ]
        ].to_dict("records")

    recommendation_id = make_hash_id(
        [
            aircraft_id,
            hazard_id,
            risk_id,
            primary_action_type,
            preferred_airport_id
        ],
        "recommendation"
    )

    recommendation_rows.append({
        "recommendation_id": recommendation_id,
        "risk_id": risk_id,
        "aircraft_id": aircraft_id,
        "hazard_id": hazard_id,
        "risk_level": risk_level_value,
        "risk_score": risk.get("risk_score"),
        "primary_action_type": primary_action_type,
        "preferred_airport_id": preferred_airport_id,
        "preferred_airport_score": preferred_airport_score,
        "candidate_airports": candidate_airports,
        "reasons": risk.get("reasons"),
        "limitations": [
            "This is an advisory recommendation only.",
            "The system does not know fuel endurance.",
            "The system does not know aircraft performance limits.",
            "The system does not know ATC clearance.",
            "The system does not know airline policy."
        ],
        "valid_from_utc": now_utc(),
        "valid_until_utc": now_utc() + timedelta(minutes=5),
        "schema_version": "recommendation.v1",
        "created_at_utc": now_utc()
    })

Recommendations = pd.DataFrame(recommendation_rows)

Recommendations.head()

,recommendation_id,risk_id,aircraft_id,hazard_id,risk_level,risk_score,primary_action_type,preferred_airport_id,preferred_airport_score,candidate_airports,reasons,limitations,valid_from_utc,valid_until_utc,schema_version,created_at_utc
0,recommendation_1f498d6ea7d0d1ef,risk_e2ee33fe41c547b3,3c4a0c,hazard_e03eae59b7b971ab,HIGH,85,EVALUATE_AVOIDANCE_OR_DIVERSION,KPVU,85.91,"[{'airport_id': 'KPVU', 'airport_name': 'Provo...",[Aircraft is currently inside the hazard polyg...,"[This is an advisory recommendation only., The...",2026-06-27 22:25:25.238003+00:00,2026-06-27 22:30:25.238005+00:00,recommendation.v1,2026-06-27 22:25:25.238010+00:00
1,recommendation_2e975e015ee69245,risk_d488c3cebf9a56e4,a9b4a4,hazard_31e97ed6e53f5ad0,HIGH,90,EVALUATE_AVOIDANCE_OR_DIVERSION,KRGA,86.33,"[{'airport_id': 'KRGA', 'airport_name': 'Richm...",[Projected centerline enters the active hazard...,"[This is an advisory recommendation only., The...",2026-06-27 22:25:25.239492+00:00,2026-06-27 22:30:25.239493+00:00,recommendation.v1,2026-06-27 22:25:25.239496+00:00
2,recommendation_a2816702f2f3edc2,risk_34a3027594d6f7a4,0d113b,hazard_92f1e5179770c450,HIGH,100,EVALUATE_AVOIDANCE_OR_DIVERSION,KLQK,86.25,"[{'airport_id': 'KLQK', 'airport_name': 'Picke...",[Aircraft is currently inside the hazard polyg...,"[This is an advisory recommendation only., The...",2026-06-27 22:25:25.240688+00:00,2026-06-27 22:30:25.240689+00:00,recommendation.v1,2026-06-27 22:25:25.240692+00:00
3,recommendation_7edd8b13fdab3d1c,risk_bec7d8b4adb1f3a2,ab6eb9,hazard_8cc4b3735942ae5e,HIGH,83,EVALUATE_AVOIDANCE_OR_DIVERSION,KCTY,84.48,"[{'airport_id': 'KCTY', 'airport_name': 'Cross...",[Projected centerline enters the active hazard...,"[This is an advisory recommendation only., The...",2026-06-27 22:25:25.241836+00:00,2026-06-27 22:30:25.241837+00:00,recommendation.v1,2026-06-27 22:25:25.241840+00:00
4,recommendation_5d31e4713b07f528,risk_1f5ff0569e46fd13,0d1130,hazard_92f1e5179770c450,HIGH,83,EVALUATE_AVOIDANCE_OR_DIVERSION,KOMH,85.08,"[{'airport_id': 'KOMH', 'airport_name': 'Orang...",[Projected centerline enters the active hazard...,"[This is an advisory recommendation only., The...",2026-06-27 22:25:25.243283+00:00,2026-06-27 22:30:25.243285+00:00,recommendation.v1,2026-06-27 22:25:25.243288+00:00


## Table 20: ActiveAlerts

In [26]:
alert_rows = []

for _, rec in Recommendations.iterrows():
    if rec.get("risk_level") not in ["HIGH", "MEDIUM"]:
        continue

    fingerprint = make_hash_id(
        [
            rec.get("aircraft_id"),
            rec.get("hazard_id"),
            rec.get("risk_level"),
            rec.get("primary_action_type"),
            rec.get("preferred_airport_id")
        ],
        "fingerprint"
    )

    alert_id = make_hash_id(
        [
            fingerprint,
            rec.get("created_at_utc")
        ],
        "alert"
    )

    alert_rows.append({
        "alert_id": alert_id,
        "fingerprint": fingerprint,
        "aircraft_id": rec.get("aircraft_id"),
        "hazard_id": rec.get("hazard_id"),
        "recommendation_id": rec.get("recommendation_id"),
        "risk_level": rec.get("risk_level"),
        "alert_type": "WEATHER_HAZARD_RISK",
        "alert_state": "NEW",
        "message": f"Aircraft {rec.get('aircraft_id')} has {rec.get('risk_level')} weather hazard risk.",
        "created_at_utc": now_utc(),
        "expires_at_utc": now_utc() + timedelta(minutes=10),
        "schema_version": "active_alert.v1"
    })

ActiveAlerts = pd.DataFrame(alert_rows)

ActiveAlerts.head()

,alert_id,fingerprint,aircraft_id,hazard_id,recommendation_id,risk_level,alert_type,alert_state,message,created_at_utc,expires_at_utc,schema_version
0,alert_c3853e74a3c15468,fingerprint_24a910397a1e303e,3c4a0c,hazard_e03eae59b7b971ab,recommendation_1f498d6ea7d0d1ef,HIGH,WEATHER_HAZARD_RISK,NEW,Aircraft 3c4a0c has HIGH weather hazard risk.,2026-06-27 22:25:25.408314+00:00,2026-06-27 22:35:25.408316+00:00,active_alert.v1
1,alert_05f2c582ea6503f4,fingerprint_117551675232b276,a9b4a4,hazard_31e97ed6e53f5ad0,recommendation_2e975e015ee69245,HIGH,WEATHER_HAZARD_RISK,NEW,Aircraft a9b4a4 has HIGH weather hazard risk.,2026-06-27 22:25:25.408393+00:00,2026-06-27 22:35:25.408393+00:00,active_alert.v1
2,alert_fe635afc2bd70818,fingerprint_71f93e36552748b0,0d113b,hazard_92f1e5179770c450,recommendation_a2816702f2f3edc2,HIGH,WEATHER_HAZARD_RISK,NEW,Aircraft 0d113b has HIGH weather hazard risk.,2026-06-27 22:25:25.408450+00:00,2026-06-27 22:35:25.408450+00:00,active_alert.v1
3,alert_0fca7ac5707d815e,fingerprint_8c038c94e6742ec8,ab6eb9,hazard_8cc4b3735942ae5e,recommendation_7edd8b13fdab3d1c,HIGH,WEATHER_HAZARD_RISK,NEW,Aircraft ab6eb9 has HIGH weather hazard risk.,2026-06-27 22:25:25.408575+00:00,2026-06-27 22:35:25.408575+00:00,active_alert.v1
4,alert_f3b06112318bd57e,fingerprint_3cf56b3cbabb4422,0d1130,hazard_92f1e5179770c450,recommendation_5d31e4713b07f528,HIGH,WEATHER_HAZARD_RISK,NEW,Aircraft 0d1130 has HIGH weather hazard risk.,2026-06-27 22:25:25.408635+00:00,2026-06-27 22:35:25.408635+00:00,active_alert.v1


In [27]:
final_tables = {
    "AircraftCurrentState": AircraftCurrentState_df,
    "ActiveHazards": ActiveHazards_df,
    "HazardCoordinates": HazardCoordinates_df,
    "StationReference": StationReference_df,
    "HazardStationCandidates": HazardStationCandidates_df,
    "MetarLatest": MetarLatest_df,
    "TafLatest": TafLatest_df,
    "TafForecastPeriods": TafForecastPeriods_df,
    "HazardCells": HazardCells,
    "AircraftProjectionPoints": AircraftProjectionPoints,
    "AircraftProjection": AircraftProjection,
    "AircraftHazardCandidates": AircraftHazardCandidates,
    "HazardEncounters": HazardEncounters,
    "RiskResults": RiskResults,
    "AirportStatus": AirportStatus,
    "AirportAssessment": AirportAssessment,
    "Recommendations": Recommendations,
    "ActiveAlerts": ActiveAlerts
}

for table_name, df in final_tables.items():
    print(f"{table_name:30s} {df.shape}")

AircraftCurrentState           (8667, 27)
ActiveHazards                  (13, 27)
HazardCoordinates              (73, 9)
StationReference               (8834, 6)
HazardStationCandidates        (1010, 11)
MetarLatest                    (676, 24)
TafLatest                      (209, 17)
TafForecastPeriods             (980, 16)
HazardCells                    (5295, 8)
AircraftProjectionPoints       (16000, 11)
AircraftProjection             (1000, 11)
AircraftHazardCandidates       (164, 12)
HazardEncounters               (164, 20)
RiskResults                    (164, 16)
AirportStatus                  (8834, 24)
AirportAssessment              (1000, 20)
Recommendations                (164, 16)
ActiveAlerts                   (164, 12)


# Wilvor Internal Schema Documentation

This section documents the local internal schemas created from the inspected APIs and the Wilvor-generated processing logic.

The notebook treats API DataFrames as simulated streaming batches. In the real cloud implementation, these tables would be populated by streaming ingestion, Lambda/Flink processing, DynamoDB current-state tables, and S3 historical storage. In this local notebook, they are represented as pandas DataFrames.

---

# 1. AircraftCurrentState

## Where it came from

`AircraftCurrentState` is created from the `opensky_df` DataFrame, which comes from the OpenSky `states/all` API.

This table represents the latest known state of each aircraft. It is a current-state operational table, meaning it is designed to keep the most recent aircraft position, speed, heading, altitude, and tracking information. In the real-time system, this table would be updated continuously as new OpenSky aircraft observations arrive.

This table is important because almost every downstream process depends on it. Aircraft projection, hazard intersection checks, risk scoring, and recommendations all start from the current aircraft state.

## Source

- API: OpenSky `states/all`
- Raw DataFrame: `opensky_df`
- Internal table: `AircraftCurrentState`

## Columns

`aircraft_id` = Unique aircraft identifier from OpenSky. This usually maps to the aircraft ICAO24 hex code.

`callsign` = Flight callsign if available. This may be empty or missing for some aircraft.

`origin_country` = Country associated with the aircraft registration according to OpenSky.

`position_time_epoch` = Original OpenSky position timestamp in Unix epoch format.

`position_time_utc` = Converted UTC timestamp showing when the aircraft position was measured.

`last_contact_epoch` = Original OpenSky timestamp showing the last time the aircraft was contacted.

`last_contact_utc` = Converted UTC timestamp for the aircraft's last contact time.

`latitude` = Current aircraft latitude.

`longitude` = Current aircraft longitude.

`baro_altitude_m` = Barometric altitude in meters as returned by OpenSky.

`geo_altitude_m` = Geometric altitude in meters as returned by OpenSky.

`baro_altitude_ft` = Barometric altitude converted from meters to feet.

`geo_altitude_ft` = Geometric altitude converted from meters to feet.

`ground_speed_mps` = Aircraft ground speed in meters per second from OpenSky.

`ground_speed_kt` = Aircraft ground speed converted to knots.

`track_deg` = Aircraft track/heading in degrees.

`vertical_rate_mps` = Aircraft vertical rate in meters per second.

`vertical_rate_fpm` = Aircraft vertical rate converted to feet per minute.

`on_ground` = Boolean flag showing whether OpenSky reports the aircraft as being on the ground.

`squawk` = Aircraft transponder squawk code if available.

`spi` = Special Position Identification flag from OpenSky.

`position_source` = Source type of the aircraft position in OpenSky.

`has_position` = Local data quality flag showing whether both latitude and longitude are available.

`source_system` = Source system name. For this table, it is `OpenSky`.

`schema_version` = Internal schema version for this table.

`received_at_utc` = Time when the local notebook processed or received this record.

`idempotency_key` = Unique deduplication key built from `aircraft_id` and `position_time_epoch`.

---

# 2. ActiveHazards

## Where it came from

`ActiveHazards` is created from the `sigmet_df` DataFrame, which comes from the Aviation Weather SIGMET/AIRMET API.

This table represents active aviation weather hazards. Each row is one weather hazard product, usually containing information such as hazard type, severity, valid time range, altitude bounds, movement, raw SIGMET text, and polygon coordinates.

This is a current-state weather hazard table. In the real-time system, this table would contain only active or recently active hazards used for aircraft-hazard detection.

## Source

- API: Aviation Weather AirSIGMET/SIGMET API
- Raw DataFrame: `sigmet_df`
- Internal table: `ActiveHazards`

## Columns

`hazard_id` = Internal unique hazard identifier generated from SIGMET fields such as series ID, creation time, valid time, and hazard type.

`source_icao_id` = Source ICAO identifier from the SIGMET product if available.

`series_id` = SIGMET series identifier from the source API.

`alpha_char` = Alpha character associated with the SIGMET product.

`receipt_time_utc` = Time when the SIGMET product was received by the source system.

`created_at_utc` = Time when the SIGMET product was created.

`valid_from_epoch` = Start time of the hazard validity window in Unix epoch format.

`valid_from_utc` = Start time of the hazard validity window converted to UTC.

`valid_to_epoch` = End time of the hazard validity window in Unix epoch format.

`valid_to_utc` = End time of the hazard validity window converted to UTC.

`product_type` = Type of aviation weather product, such as SIGMET or AIRMET.

`hazard_type` = Type of hazard, such as turbulence, icing, IFR, convection, mountain obscuration, or other aviation weather hazard.

`severity` = Severity label from the source product.

`lower_altitude_1_ft` = First lower altitude bound for the hazard in feet.

`upper_altitude_1_ft` = First upper altitude bound for the hazard in feet.

`lower_altitude_2_ft` = Second lower altitude bound if the product contains another altitude range.

`upper_altitude_2_ft` = Second upper altitude bound if the product contains another altitude range.

`movement_direction_deg` = Movement direction of the hazard in degrees if provided.

`movement_speed_kt` = Movement speed of the hazard in knots if provided.

`polygon_coords` = Original polygon coordinate list from the SIGMET product.

`polygon_point_count` = Number of coordinate points in the hazard polygon.

`raw_text` = Original raw SIGMET/AIRMET text.

`post_process_flag` = Source flag showing whether post-processing was applied.

`status` = Internal status of the hazard. In this notebook it is set to `ACTIVE`.

`source_system` = Source system name. For this table, it is `NOAA_AviationWeather_SIGMET`.

`schema_version` = Internal schema version for this table.

`received_at_utc` = Time when the local notebook processed or received this hazard record.

---

# 3. HazardCoordinates

## Where it came from

`HazardCoordinates` is generated from `ActiveHazards`.

The SIGMET API returns polygon coordinates as a nested list. This table explodes those nested coordinates into one row per polygon point. It is useful for debugging geometry, visualizing hazard polygons, and later storing polygon points in a structured way.

## Source

- Source table: `ActiveHazards`
- Original API: Aviation Weather SIGMET/AIRMET API
- Internal table: `HazardCoordinates`

## Columns

`hazard_id` = Internal hazard identifier linking this coordinate point back to `ActiveHazards`.

`sequence_number` = Order of the coordinate point within the polygon.

`latitude` = Latitude of this polygon coordinate point.

`longitude` = Longitude of this polygon coordinate point.

`valid_from_utc` = Start time of the hazard validity window.

`valid_to_utc` = End time of the hazard validity window.

`hazard_type` = Type of hazard associated with this polygon.

`severity` = Severity of the hazard associated with this polygon.

`schema_version` = Internal schema version for this table.

---

# 4. StationReference

## Where it came from

`StationReference` is created from the Aviation Weather station cache.

This table is not METAR or TAF data. It is a station lookup table. Its job is to translate geography into weather station IDs. Since SIGMET gives hazard polygons but METAR and TAF APIs require station IDs, this table allows Wilvor to find which stations are inside or near a SIGMET polygon.

This is a reference table, not a streaming event table.

## Source

- Source: Aviation Weather station cache
- Internal table: `StationReference`

## Columns

`station_id` = Weather station identifier, usually an ICAO station code such as `KJFK`, `KBOS`, or `KORD`.

`station_name` = Human-readable station or airport name if available.

`latitude` = Station latitude.

`longitude` = Station longitude.

`elevation_m` = Station elevation in meters if available.

`schema_version` = Internal schema version for this table.

---

# 5. HazardStationCandidates

## Where it came from

`HazardStationCandidates` is generated by combining `ActiveHazards` and `StationReference`.

This table answers the question:

Which weather stations are inside or near each active SIGMET polygon?

This is the bridge between SIGMET and METAR/TAF. SIGMET does not directly give METAR/TAF station IDs, so this table derives the station IDs that should be passed into the METAR and TAF APIs.

## Source

- Source table 1: `ActiveHazards`
- Source table 2: `StationReference`
- Internal table: `HazardStationCandidates`

## Columns

`hazard_id` = Internal hazard identifier from `ActiveHazards`.

`hazard_type` = Type of hazard associated with the SIGMET.

`severity` = Severity of the hazard.

`valid_from_utc` = Start time of the hazard validity window.

`valid_to_utc` = End time of the hazard validity window.

`station_id` = Weather station ID found inside or near the hazard polygon.

`station_name` = Name of the station or airport.

`station_latitude` = Latitude of the station.

`station_longitude` = Longitude of the station.

`reason` = Reason this station was selected. In this notebook, the value is `STATION_INSIDE_OR_NEAR_SIGMET`.

`schema_version` = Internal schema version for this table.

---

# 6. METAR

## Where it came from

`METAR` or `metar_df` is the raw API response from the Aviation Weather METAR endpoint.

This table is not the final internal schema. It is the raw or semi-raw API result fetched using station IDs derived from `HazardStationCandidates`.

In the real system, this would be treated as an incoming weather batch. It would be validated, normalized, and then written into `MetarLatest`.

## Source

- API: `https://aviationweather.gov/api/data/metar`
- Parameter: comma-separated `ids`
- IDs came from: `HazardStationCandidates`
- Raw DataFrame: `metar_df`

## Main fields used from the API

`icaoId` = Station identifier returned by the METAR API.

`name` = Station or airport name.

`obsTime` = Observation time in Unix epoch format.

`receiptTime` = Time when the METAR was received by the weather system.

`temp` = Temperature in Celsius.

`dewp` = Dew point in Celsius.

`wdir` = Wind direction in degrees.

`wspd` = Wind speed in knots.

`wgst` = Wind gust speed in knots.

`visib` = Visibility in statute miles.

`altim` = Altimeter setting.

`slp` = Sea level pressure.

`wxString` = Weather string containing weather codes such as rain, mist, thunderstorms, snow, etc.

`precip` = Precipitation value if available.

`fltCat` = Flight category such as VFR, MVFR, IFR, or LIFR.

`metarType` = METAR report type.

`lat` = Station latitude.

`lon` = Station longitude.

`elev` = Station elevation.

`clouds` = Nested cloud layer information.

`rawOb` = Original raw METAR text.

---

# 7. MetarLatest

## Where it came from

`MetarLatest` is created from the raw `metar_df`.

This table stores the latest normalized METAR observation per station. It is a current airport-weather table. In the real-time system, this table would be updated whenever a new METAR is fetched for a station affected by a hazard or needed for airport evaluation.

## Source

- Source table: `metar_df`
- Original API: Aviation Weather METAR API
- Internal table: `MetarLatest`

## Columns

`station_id` = Weather station identifier.

`station_name` = Station or airport name.

`observed_time_epoch` = METAR observation time in Unix epoch format.

`observed_time_utc` = METAR observation time converted to UTC.

`receipt_time_utc` = Time when the METAR was received by the weather system.

`temperature_c` = Temperature in Celsius.

`dewpoint_c` = Dew point in Celsius.

`wind_direction_deg` = Wind direction in degrees.

`wind_speed_kt` = Wind speed in knots.

`wind_gust_kt` = Wind gust speed in knots.

`visibility_sm` = Visibility in statute miles.

`altimeter_hpa` = Altimeter value from the METAR API.

`sea_level_pressure_hpa` = Sea level pressure if available.

`weather_string` = Weather condition string from the METAR, such as rain, mist, thunderstorm, or snow codes.

`precipitation_in` = Precipitation amount if provided.

`flight_category` = Flight category such as VFR, MVFR, IFR, or LIFR.

`metar_type` = Type of METAR observation.

`latitude` = Station latitude.

`longitude` = Station longitude.

`elevation_m` = Station elevation in meters.

`clouds` = Nested cloud layer list from the METAR response.

`raw_text` = Original raw METAR text.

`source_system` = Source system name. For this table, it is `NOAA_AviationWeather_METAR`.

`schema_version` = Internal schema version for this table.

---

# 8. TAF

## Where it came from

`TAF` or `taf_df` is the raw API response from the Aviation Weather TAF endpoint.

This table is not the final internal schema. It is the raw or semi-raw API result fetched using station IDs derived from the SIGMET impact area. Later, in the full Wilvor workflow, TAF should be fetched mainly for candidate diversion airports because TAF is used to understand forecast conditions around estimated arrival time.

## Source

- API: `https://aviationweather.gov/api/data/taf`
- Parameter: comma-separated `ids`
- IDs came from: `HazardStationCandidates`
- Raw DataFrame: `taf_df`

## Main fields used from the API

`icaoId` = Station identifier returned by the TAF API.

`name` = Station or airport name.

`issueTime` = Time when the TAF was issued.

`bulletinTime` = Bulletin publication time.

`validTimeFrom` = Start time of the TAF validity window.

`validTimeTo` = End time of the TAF validity window.

`mostRecent` = Flag showing whether this is the most recent TAF for the station.

`remarks` = TAF remarks if available.

`lat` = Station latitude.

`lon` = Station longitude.

`elev` = Station elevation.

`rawTAF` = Original raw TAF text.

`fcsts` = Nested list of forecast periods inside the TAF.

---

# 9. TafLatest

## Where it came from

`TafLatest` is created from raw `taf_df`.

This table stores the latest normalized TAF header per station. It does not fully represent the forecast details by itself because TAF contains multiple forecast periods. Those periods are stored separately in `TafForecastPeriods`.

## Source

- Source table: `taf_df`
- Original API: Aviation Weather TAF API
- Internal table: `TafLatest`

## Columns

`station_id` = Weather station identifier.

`station_name` = Station or airport name.

`issued_at_utc` = Time when the TAF was issued.

`bulletin_time_utc` = Bulletin publication time.

`valid_from_epoch` = Start time of the TAF validity window in Unix epoch format.

`valid_from_utc` = Start time of the TAF validity window converted to UTC.

`valid_to_epoch` = End time of the TAF validity window in Unix epoch format.

`valid_to_utc` = End time of the TAF validity window converted to UTC.

`most_recent` = Flag showing whether this is the most recent TAF.

`remarks` = TAF remarks if available.

`latitude` = Station latitude.

`longitude` = Station longitude.

`elevation_m` = Station elevation in meters.

`raw_text` = Original raw TAF text.

`fcsts` = Nested forecast-period list from the original TAF response.

`source_system` = Source system name. For this table, it is `NOAA_AviationWeather_TAF`.

`schema_version` = Internal schema version for this table.

---

# 10. TafForecastPeriods

## Where it came from

`TafForecastPeriods` is created by exploding the nested `fcsts` list from `taf_df`.

This table is very important because TAF is not a single weather row. A TAF contains multiple forecast periods, and each period may describe a different future condition. Wilvor needs this table to answer:

What will the weather look like at a candidate airport around the estimated arrival time?

## Source

- Source table: `taf_df`
- Nested source field: `fcsts`
- Internal table: `TafForecastPeriods`

## Columns

`station_id` = Weather station identifier.

`issued_at_utc` = TAF issue time.

`period_from_epoch` = Forecast period start time in Unix epoch format.

`period_from_utc` = Forecast period start time converted to UTC.

`period_to_epoch` = Forecast period end time in Unix epoch format.

`period_to_utc` = Forecast period end time converted to UTC.

`change_type` = Forecast change indicator, such as base forecast, temporary condition, or becoming condition.

`probability` = Probability value if the forecast period has a probabilistic condition.

`wind_direction_deg` = Forecast wind direction in degrees.

`wind_speed_kt` = Forecast wind speed in knots.

`wind_gust_kt` = Forecast wind gust speed in knots.

`visibility_sm` = Forecast visibility in statute miles.

`weather_string` = Forecast weather condition string.

`clouds` = Nested cloud information for the forecast period.

`sequence_number` = Order of the forecast period within the TAF.

`schema_version` = Internal schema version for this table.

---

# 11. HazardCells

## Where it came from

`HazardCells` is generated from `ActiveHazards`.

This table converts each SIGMET polygon into H3 spatial cells. It is used for fast spatial lookup. Instead of comparing every aircraft projection with every SIGMET polygon, Wilvor first checks whether aircraft projection cells overlap hazard cells.

This table does not replace exact geometry checks. It only narrows down the possible candidates.

## Source

- Source table: `ActiveHazards`
- Processing logic: H3 polygon cell conversion
- Internal table: `HazardCells`

## Columns

`h3_cell` = H3 spatial index cell covering part of the hazard polygon.

`hazard_id` = Internal hazard identifier from `ActiveHazards`.

`hazard_type` = Type of hazard associated with the H3 cell.

`severity` = Severity of the hazard.

`valid_from_utc` = Start time of the hazard validity window.

`valid_to_utc` = End time of the hazard validity window.

`schema_version` = Internal schema version for this table.

`created_at_utc` = Time when this H3 mapping was generated.

---

# 12. AircraftProjectionPoints

## Where it came from

`AircraftProjectionPoints` is generated from `AircraftCurrentState`.

This table projects each aircraft forward using its current position, speed, track, and vertical rate. In this notebook, it is a simplified motion projection. In the full system, this would become the basis for projected path, uncertainty corridor, hazard lookup, and risk scoring.

Each row represents one projected aircraft point at a future time horizon.

## Source

- Source table: `AircraftCurrentState`
- Processing logic: Local trajectory projection
- Internal table: `AircraftProjectionPoints`

## Columns

`projection_id` = Internal unique identifier for one aircraft projection run.

`aircraft_id` = Aircraft identifier linked to `AircraftCurrentState`.

`generated_at_utc` = Time when the projection was generated.

`horizon_min` = Number of minutes into the future for this projected point.

`projected_time_utc` = UTC timestamp for this projected point.

`latitude` = Projected aircraft latitude.

`longitude` = Projected aircraft longitude.

`estimated_altitude_ft` = Estimated aircraft altitude at this projected point.

`confidence` = Projection confidence label. Shorter horizons have higher confidence; longer horizons have lower confidence.

`h3_cell` = H3 cell for the projected aircraft point.

`schema_version` = Internal schema version for this table.

---

# 13. AircraftProjection

## Where it came from

`AircraftProjection` is generated from `AircraftProjectionPoints`.

This table summarizes the full aircraft trajectory projection into one row per aircraft projection. It stores the projected points, uncertainty corridor, corridor H3 cells, and validity period.

This table is used to compare aircraft future movement against active hazards.

## Source

- Source table: `AircraftProjectionPoints`
- Processing logic: Trajectory path and uncertainty corridor generation
- Internal table: `AircraftProjection`

## Columns

`projection_id` = Internal unique identifier for the aircraft projection.

`aircraft_id` = Aircraft identifier linked to `AircraftCurrentState`.

`generated_at_utc` = Time when the projection was generated.

`valid_until_utc` = Time until which this projection should be considered usable.

`projection_horizon_min` = Maximum projection horizon in minutes.

`projection_points` = List of projected points for the aircraft path.

`corridor_half_width_nm` = Half-width of the uncertainty corridor in nautical miles.

`corridor_polygon_geojson` = GeoJSON representation of the buffered projected corridor.

`corridor_h3_cells` = List of H3 cells covered by the uncertainty corridor.

`corridor_h3_cell_count` = Number of H3 cells in the projected corridor.

`schema_version` = Internal schema version for this table.

---

# 14. AircraftHazardCandidates

## Where it came from

`AircraftHazardCandidates` is generated by joining `AircraftProjection` with `HazardCells`.

This is the first merge between aircraft movement and weather hazards. It identifies aircraft projections whose corridor H3 cells overlap active hazard H3 cells.

This table does not confirm an actual intersection. It only identifies possible aircraft-hazard matches that need exact geometry evaluation.

## Source

- Source table 1: `AircraftProjection`
- Source table 2: `HazardCells`
- Processing logic: H3 cell overlap
- Internal table: `AircraftHazardCandidates`

## Columns

`projection_id` = Aircraft projection identifier.

`aircraft_id` = Aircraft identifier.

`generated_at_utc` = Time when the aircraft projection was generated.

`h3_cell` = Shared H3 cell between the aircraft projection corridor and hazard polygon.

`hazard_id` = Internal hazard identifier.

`hazard_type` = Type of hazard.

`severity` = Hazard severity.

`valid_from_utc` = Start time of the hazard validity window.

`valid_to_utc` = End time of the hazard validity window.

`candidate_reason` = Reason this aircraft-hazard pair was selected. In this notebook, it is `PROJECTION_CORRIDOR_CELL_OVERLAPS_HAZARD_CELL`.

`schema_version` = Internal schema version for this table.

`created_at_utc` = Time when the candidate record was created.

---

# 15. HazardEncounters

## Where it came from

`HazardEncounters` is generated from `AircraftHazardCandidates`, `AircraftProjection`, and `ActiveHazards`.

This table performs the exact geometry stage. H3 only finds possible candidates. This table checks whether the aircraft projection corridor actually intersects the SIGMET polygon and whether the aircraft altitude overlaps the hazard altitude band.

This is one of the most important operational intelligence tables because it turns raw aircraft movement and raw weather hazards into a meaningful aircraft-hazard encounter.

## Source

- Source table 1: `AircraftHazardCandidates`
- Source table 2: `AircraftProjection`
- Source table 3: `ActiveHazards`
- Processing logic: Exact geometry and altitude overlap checks
- Internal table: `HazardEncounters`

## Columns

`encounter_id` = Internal unique identifier for the aircraft-hazard encounter.

`aircraft_id` = Aircraft identifier.

`projection_id` = Aircraft projection identifier.

`hazard_id` = Hazard identifier.

`hazard_type` = Type of hazard involved in the encounter.

`severity` = Hazard severity.

`detected_at_utc` = Time when the encounter was detected.

`corridor_intersects` = Boolean showing whether the aircraft uncertainty corridor intersects the hazard polygon.

`centerline_intersects` = Boolean showing whether the projected aircraft centerline enters the hazard polygon.

`inside_now` = Boolean showing whether the aircraft appears to already be inside the hazard polygon.

`first_intersection_horizon_min` = Estimated number of minutes until the first projected intersection or closest approach.

`first_intersection_time_utc` = Estimated UTC time of first intersection or closest approach.

`altitude_at_intersection_ft` = Estimated aircraft altitude at the intersection or closest point.

`hazard_lower_altitude_ft` = Lower altitude bound of the hazard.

`hazard_upper_altitude_ft` = Upper altitude bound of the hazard.

`altitude_overlap` = Boolean or unknown value showing whether the aircraft altitude overlaps the hazard altitude band.

`closest_distance_degrees` = Approximate geometry distance to the hazard polygon in coordinate degrees.

`valid_from_utc` = Start time of the hazard validity window.

`valid_to_utc` = End time of the hazard validity window.

`schema_version` = Internal schema version for this table.

---

# 16. RiskResults

## Where it came from

`RiskResults` is generated from `HazardEncounters`.

This table converts aircraft-hazard encounters into explainable risk scores. The score is based on hazard type/severity, geometry, time to intersection, and altitude overlap.

This table is important because Wilvor should not only say that an aircraft and hazard are near each other. It should explain how serious the situation is and why.

## Source

- Source table: `HazardEncounters`
- Processing logic: Rule-based risk scoring
- Internal table: `RiskResults`

## Columns

`risk_id` = Internal unique identifier for the risk result.

`encounter_id` = Encounter identifier from `HazardEncounters`.

`aircraft_id` = Aircraft identifier.

`hazard_id` = Hazard identifier.

`hazard_type` = Type of hazard involved in the risk result.

`severity` = Hazard severity.

`risk_score` = Final numerical risk score from 0 to 100.

`risk_level` = Risk classification such as LOW, MEDIUM, or HIGH.

`hazard_component_score` = Score contribution from hazard type and severity.

`geometry_component_score` = Score contribution from geometry relationship, such as corridor or centerline intersection.

`time_component_score` = Score contribution from time to intersection.

`altitude_component_score` = Score contribution from altitude overlap.

`confidence` = Confidence label for the risk result.

`reasons` = Human-readable list explaining why the score was assigned.

`generated_at_utc` = Time when the risk result was generated.

`schema_version` = Internal schema version for this table.

---

# 17. AirportStatus

## Where it came from

`AirportStatus` is generated by combining `StationReference`, `MetarLatest`, and `TafLatest`.

This table gives Wilvor a current operational weather view of each airport or station. It combines current observation data from METAR with forecast header data from TAF, then classifies the airport weather risk.

This table is used later by airport assessment and diversion recommendation logic.

## Source

- Source table 1: `StationReference`
- Source table 2: `MetarLatest`
- Source table 3: `TafLatest`
- Processing logic: Weather status classification
- Internal table: `AirportStatus`

## Columns

`airport_id` = Airport or weather station identifier.

`airport_name` = Airport or station name.

`latitude` = Airport or station latitude.

`longitude` = Airport or station longitude.

`elevation_m` = Airport or station elevation in meters.

`observed_time_utc` = Latest METAR observation time.

`temperature_c` = Latest observed temperature in Celsius.

`dewpoint_c` = Latest observed dew point in Celsius.

`wind_direction_deg` = Latest observed wind direction in degrees.

`wind_speed_kt` = Latest observed wind speed in knots.

`wind_gust_kt` = Latest observed wind gust speed in knots.

`visibility_sm` = Latest observed visibility in statute miles.

`weather_string` = Latest observed weather condition string.

`flight_category` = Latest flight category from METAR.

`raw_text_metar` = Raw METAR text after merging into airport status.

`issued_at_utc` = Latest TAF issue time.

`valid_from_utc` = TAF validity start time.

`valid_to_utc` = TAF validity end time.

`raw_text_taf` = Raw TAF text after merging into airport status.

`weather_risk_level` = Internal weather risk classification for the airport, such as LOW, MEDIUM, HIGH, or UNKNOWN.

`has_metar` = Boolean flag showing whether a METAR record exists for this airport.

`has_taf` = Boolean flag showing whether a TAF record exists for this airport.

`schema_version` = Internal schema version for this table.

`updated_at_utc` = Time when this airport status record was generated or updated.

---

# 18. AirportAssessment

## Where it came from

`AirportAssessment` is generated from `RiskResults`, `AircraftCurrentState`, and `AirportStatus`.

This table evaluates possible airports for aircraft affected by MEDIUM or HIGH risk encounters. In the current notebook, this is a simplified local version. It uses distance, weather risk, and placeholder scores for route safety, congestion, and runway suitability.

In the real Wilvor system, this table would become more advanced by using route hazard checks, runway data, airport closure status, traffic congestion, and aircraft-specific operational constraints.

## Source

- Source table 1: `RiskResults`
- Source table 2: `AircraftCurrentState`
- Source table 3: `AirportStatus`
- Processing logic: Candidate airport scoring
- Internal table: `AirportAssessment`

## Columns

`airport_assessment_id` = Internal unique identifier for the airport assessment.

`risk_id` = Risk result identifier linked to `RiskResults`.

`aircraft_id` = Aircraft identifier.

`hazard_id` = Hazard identifier associated with the aircraft risk.

`airport_id` = Candidate airport or station identifier.

`airport_name` = Candidate airport or station name.

`airport_latitude` = Candidate airport latitude.

`airport_longitude` = Candidate airport longitude.

`distance_nm` = Distance from aircraft to candidate airport in nautical miles.

`eta_minutes` = Estimated time to reach the candidate airport in minutes.

`weather_risk_level` = Current weather risk level for the candidate airport.

`weather_score` = Score contribution based on airport weather suitability.

`distance_score` = Score contribution based on distance from the aircraft.

`hazard_free_route_score` = Placeholder score for whether the route to the airport avoids hazards.

`congestion_score` = Placeholder score for estimated airport congestion.

`runway_score` = Placeholder score for runway suitability.

`total_airport_score` = Final weighted score for the candidate airport.

`known_limitations` = List of known missing operational factors, such as fuel, runway suitability, aircraft performance, active runway, and airline policy.

`schema_version` = Internal schema version for this table.

`created_at_utc` = Time when the airport assessment was created.

---

# 19. Recommendations

## Where it came from

`Recommendations` is generated from `RiskResults` and `AirportAssessment`.

This table creates the advisory decision-support output. It does not command an aircraft to do anything. Instead, it gives a primary advisory action, candidate airport options, reasons, limitations, and validity period.

This table is one of the main outputs of Wilvor's decision intelligence layer.

## Source

- Source table 1: `RiskResults`
- Source table 2: `AirportAssessment`
- Processing logic: Recommendation generation
- Internal table: `Recommendations`

## Columns

`recommendation_id` = Internal unique identifier for the recommendation.

`risk_id` = Risk result identifier linked to `RiskResults`.

`aircraft_id` = Aircraft identifier.

`hazard_id` = Hazard identifier.

`risk_level` = Risk level that triggered the recommendation.

`risk_score` = Numerical risk score.

`primary_action_type` = Main advisory action, such as MONITOR, MONITOR_AND_PREPARE_OPTIONS, or EVALUATE_AVOIDANCE_OR_DIVERSION.

`preferred_airport_id` = Highest-ranked candidate airport if available.

`preferred_airport_score` = Score of the highest-ranked candidate airport.

`candidate_airports` = List of ranked candidate airports and supporting values.

`reasons` = Human-readable reasons inherited from the risk result.

`limitations` = Known limitations that must be shown to users, such as missing fuel, aircraft performance, ATC clearance, and airline policy.

`valid_from_utc` = Start time of recommendation validity.

`valid_until_utc` = End time of recommendation validity.

`schema_version` = Internal schema version for this table.

`created_at_utc` = Time when the recommendation was created.

---

# 20. ActiveAlerts

## Where it came from

`ActiveAlerts` is generated from `Recommendations`.

This table manages alert state and deduplication. It only creates alerts for MEDIUM or HIGH risk recommendations. The fingerprint prevents the same aircraft-hazard recommendation from repeatedly alerting users when nothing materially changed.

In the real system, this table would support alert lifecycle states such as NEW, UPDATED, ESCALATED, RESOLVED, and EXPIRED.

## Source

- Source table: `Recommendations`
- Processing logic: Alert creation and deduplication
- Internal table: `ActiveAlerts`

## Columns

`alert_id` = Internal unique identifier for the alert.

`fingerprint` = Deduplication key generated from aircraft, hazard, risk level, action type, and preferred airport.

`aircraft_id` = Aircraft identifier.

`hazard_id` = Hazard identifier.

`recommendation_id` = Recommendation identifier linked to `Recommendations`.

`risk_level` = Risk level associated with the alert.

`alert_type` = Type of alert. In this notebook, it is `WEATHER_HAZARD_RISK`.

`alert_state` = Current alert state. In this notebook, new alerts are set to `NEW`.

`message` = Human-readable alert message.

`created_at_utc` = Time when the alert was created.

`expires_at_utc` = Time when the alert should expire.

`schema_version` = Internal schema version for this table.